In [3]:
"""
Qwen 파인튜닝 데이터셋 토큰 수 분석
- train/eval JSONL 파일의 토큰 수 분포를 확인
- Qwen3.5 토크나이저 사용 (정확한 토큰 수)
"""

import json
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer

# -------------------------
# Config
# -------------------------
MODEL_ID = "Qwen/Qwen3.5-9B"
TRAIN_PATH = "./data/7_qwen_dataset_train.jsonl"
EVAL_PATH = "./data/7_qwen_dataset_eval.jsonl"
MAX_SEQ_LENGTH = 131_072

# -------------------------
# 토크나이저 로드
# -------------------------
print("📦 토크나이저 로딩...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)


def count_tokens_for_sample(sample: dict, tokenizer) -> dict:
    """prompt(system+user) / completion(assistant) 토큰 수 분리 계산"""
    msgs = sample["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]

    # prompt: system + user + generation prompt
    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    completion_text = asst_msgs[0]["content"] + "<|im_end|>\n"

    p_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    c_ids = tokenizer.encode(completion_text, add_special_tokens=False)

    return {
        "prompt_tokens": len(p_ids),
        "completion_tokens": len(c_ids),
        "total_tokens": len(p_ids) + len(c_ids),
        "channel": sample.get("metadata", {}).get("channel", ""),
        "video": sample.get("metadata", {}).get("video", ""),
        "num_instances": sample.get("metadata", {}).get("num_instances", 0),
    }


def analyze_file(path: str, label: str):
    """JSONL 파일 토큰 수 분석"""
    print(f"\n{'='*60}")
    print(f"📂 {label}: {path}")
    print(f"{'='*60}")

    samples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            samples.append(json.loads(line))
    print(f"  총 {len(samples):,}개 샘플")

    # 토큰 수 계산
    results = []
    for s in tqdm(samples, desc="토큰 수 계산"):
        results.append(count_tokens_for_sample(s, tokenizer))

    prompt_tokens = np.array([r["prompt_tokens"] for r in results])
    completion_tokens = np.array([r["completion_tokens"] for r in results])
    total_tokens = np.array([r["total_tokens"] for r in results])

    # ---- 기본 통계 ----
    print(f"\n📊 토큰 수 통계")
    print(f"{'':>20} {'prompt':>12} {'completion':>12} {'total':>12}")
    print(f"  {'─'*56}")
    for name, arr in [("prompt", prompt_tokens), ("completion", completion_tokens), ("total", total_tokens)]:
        print(f"  {'mean':>18} {'-':>12} {'-':>12} {arr.mean():>12,.0f}" if name != "total" else "", end="")
    # 깔끔하게 다시
    print()
    for stat_name, func in [
        ("mean", np.mean), ("median", np.median), ("std", np.std),
        ("min", np.min), ("max", np.max),
        ("p5", lambda x: np.percentile(x, 5)),
        ("p25", lambda x: np.percentile(x, 25)),
        ("p75", lambda x: np.percentile(x, 75)),
        ("p90", lambda x: np.percentile(x, 90)),
        ("p95", lambda x: np.percentile(x, 95)),
        ("p99", lambda x: np.percentile(x, 99)),
    ]:
        p = func(prompt_tokens)
        c = func(completion_tokens)
        t = func(total_tokens)
        print(f"  {stat_name:>18} {p:>12,.0f} {c:>12,.0f} {t:>12,.0f}")

    # ---- MAX_SEQ_LENGTH 초과 ----
    over = (total_tokens > MAX_SEQ_LENGTH).sum()
    print(f"\n⚠️  MAX_SEQ_LENGTH({MAX_SEQ_LENGTH:,}) 초과: {over:,}개 ({over/len(results)*100:.2f}%)")

    # ---- 구간별 분포 ----
    bins = [0, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, float("inf")]
    labels = ["~1K", "1K~2K", "2K~4K", "4K~8K", "8K~16K", "16K~32K", "32K~64K", "64K~128K", "128K+"]
    print(f"\n📈 total 토큰 구간별 분포")
    for i in range(len(bins) - 1):
        cnt = ((total_tokens >= bins[i]) & (total_tokens < bins[i + 1])).sum()
        pct = cnt / len(results) * 100
        bar = "█" * int(pct / 2)
        print(f"  {labels[i]:>10}: {cnt:>7,}개 ({pct:>5.1f}%) {bar}")

    # ---- 텔롭 없는 영상 ----
    n_empty = sum(1 for r in results if r["num_instances"] == 0)
    print(f"\n📝 텔롭 없는 영상(instances=0): {n_empty:,}개 ({n_empty/len(results)*100:.1f}%)")

    # ---- 가장 긴 샘플 top 10 ----
    sorted_idx = np.argsort(total_tokens)[::-1]
    print(f"\n🔝 가장 긴 샘플 Top 10")
    print(f"  {'#':>4} {'total':>10} {'prompt':>10} {'completion':>10}  채널/영상")
    for rank, idx in enumerate(sorted_idx[:10], 1):
        r = results[idx]
        print(f"  {rank:>4} {r['total_tokens']:>10,} {r['prompt_tokens']:>10,} {r['completion_tokens']:>10,}  {r['channel']}/{r['video'][:40]}")

    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    import os

    train_results = analyze_file(TRAIN_PATH, "Train")

    if os.path.exists(EVAL_PATH):
        eval_results = analyze_file(EVAL_PATH, "Eval")

    # ---- 전체 합산 요약 ----
    all_total = np.array([r["total_tokens"] for r in train_results])
    if os.path.exists(EVAL_PATH):
        all_total = np.concatenate([all_total, np.array([r["total_tokens"] for r in eval_results])])

    total_training_tokens = all_total.sum()
    print(f"\n{'='*60}")
    print(f"📊 전체 요약")
    print(f"  총 학습 토큰 수: {total_training_tokens:,.0f} ({total_training_tokens/1e9:.2f}B)")
    print(f"  train 총 토큰: {np.array([r['total_tokens'] for r in train_results]).sum():,.0f}")
    print(f"{'='*60}")

📦 토크나이저 로딩...

📂 Train: ./data/7_qwen_dataset_train.jsonl
  총 63,080개 샘플


토큰 수 계산: 100%|██████████| 63080/63080 [02:15<00:00, 466.68it/s] 



📊 토큰 수 통계
                           prompt   completion        total
  ────────────────────────────────────────────────────────
                mean            -            -          487                mean            -            -        1,327
                mean          487        1,327        1,815
              median          430          778        1,242
                 std          229        2,231        2,333
                 min          220            6          233
                 max        3,252       92,949       93,511
                  p5          246            6          281
                 p25          305          116          489
                 p75          614        1,725        2,345
                 p90          784        3,088        3,755
                 p95          890        4,412        5,127
                 p99        1,234        8,896        9,666

⚠️  MAX_SEQ_LENGTH(131,072) 초과: 0개 (0.00%)

📈 total 토큰 구간별 분포
         ~1K:  28,110개 ( 44.

토큰 수 계산: 100%|██████████| 3320/3320 [00:07<00:00, 423.05it/s]


📊 토큰 수 통계
                           prompt   completion        total
  ────────────────────────────────────────────────────────
                mean            -            -          491                mean            -            -        1,334
                mean          491        1,334        1,825
              median          434          764        1,232
                 std          234        2,154        2,260
                 min          228            6          241
                 max        2,186       47,563       48,329
                  p5          247            6          287
                 p25          306          117          484
                 p75          620        1,699        2,330
                 p90          788        3,147        3,833
                 p95          906        4,437        5,186
                 p99        1,303        9,001        9,951

⚠️  MAX_SEQ_LENGTH(131,072) 초과: 0개 (0.00%)

📈 total 토큰 구간별 분포
         ~1K:   1,475개 ( 44.

In [1]:
import os
import glob
from datetime import datetime

OUTPUT_DIR = "./data/7_qwen_test"

# 모든 json 파일 재귀 탐색
json_files = glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True)

if not json_files:
    print("❌ json 파일이 없습니다.")
else:
    # mtime(수정 시간) 기준으로 가장 최근 파일 찾기
    latest = max(json_files, key=os.path.getmtime)
    mtime = datetime.fromtimestamp(os.path.getmtime(latest))
    size_kb = os.path.getsize(latest) / 1024

    print(f"📄 가장 최근 파일: {latest}")
    print(f"   수정 시간: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   크기: {size_kb:.1f} KB")
    print(f"   총 json 파일 수: {len(json_files)}")

📄 가장 최근 파일: ./data/7_qwen_test/AKMU/이찬혁 (LEE CHANHYUK) - '1조 (1 TRILLION)' M⧸V OUT NOW/AKMU.json
   수정 시간: 2026-04-21 14:18:57
   크기: 20.4 KB
   총 json 파일 수: 3038


In [2]:
# %% 데이터셋 토큰 수 분석
import json
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer

# -------------------------
# Config
# -------------------------
MODEL_NAME = "unsloth/Qwen3.5-9B"  # 실제 파인튜닝에 쓰는 모델로 맞춰주세요
DATASET_PATHS = [
    "./data/7_qwen_dataset_train.jsonl",
    "./data/7_qwen_dataset_eval.jsonl",
]
MAX_SEQ_LENGTH = 8192  # 참고용 임계값

# -------------------------
# 토크나이저 로드
# -------------------------
print(f"📦 토크나이저 로딩: {MODEL_NAME}")
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# -------------------------
# 토큰 수 계산
# -------------------------
for path in DATASET_PATHS:
    print(f"\n{'='*60}")
    print(f"📂 {path}")
    print('='*60)

    samples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            samples.append(json.loads(line))
    print(f"  샘플 수: {len(samples):,}")

    n_tokens_list = []
    n_prompt_list = []
    n_completion_list = []
    top_samples = []  # (n_tokens, idx, meta)

    for i, sample in enumerate(tqdm(samples, desc="토큰 계산")):
        msgs = sample["messages"]
        prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
        asst_msgs = [m for m in msgs if m["role"] == "assistant"]

        prompt_text = tok.apply_chat_template(
            prompt_msgs, tokenize=False, add_generation_prompt=True
        )
        completion_text = asst_msgs[0]["content"] + "<|im_end|>\n" if asst_msgs else ""

        n_p = len(tok.encode(prompt_text, add_special_tokens=False))
        n_c = len(tok.encode(completion_text, add_special_tokens=False))
        n_total = n_p + n_c

        n_prompt_list.append(n_p)
        n_completion_list.append(n_c)
        n_tokens_list.append(n_total)

        # 상위 10개 추적 (메모리 절약)
        if len(top_samples) < 10:
            top_samples.append((n_total, i, sample))
            top_samples.sort(key=lambda x: -x[0])
        elif n_total > top_samples[-1][0]:
            top_samples[-1] = (n_total, i, sample)
            top_samples.sort(key=lambda x: -x[0])

    arr = np.array(n_tokens_list)
    p_arr = np.array(n_prompt_list)
    c_arr = np.array(n_completion_list)

    print(f"\n📊 전체 토큰 수 (prompt + completion)")
    print(f"  min   : {arr.min():>9,}")
    print(f"  mean  : {arr.mean():>9,.0f}")
    print(f"  median: {int(np.median(arr)):>9,}")
    print(f"  p90   : {int(np.percentile(arr, 90)):>9,}")
    print(f"  p95   : {int(np.percentile(arr, 95)):>9,}")
    print(f"  p99   : {int(np.percentile(arr, 99)):>9,}")
    print(f"  p99.9 : {int(np.percentile(arr, 99.9)):>9,}")
    print(f"  max   : {arr.max():>9,}")

    print(f"\n📊 prompt / completion 분리")
    print(f"  prompt      max: {p_arr.max():>9,}  mean: {p_arr.mean():>9,.0f}")
    print(f"  completion  max: {c_arr.max():>9,}  mean: {c_arr.mean():>9,.0f}")

    print(f"\n📊 MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} 기준")
    over = int((arr > MAX_SEQ_LENGTH).sum())
    print(f"  초과: {over:,}개 ({over/len(arr)*100:.2f}%)")
    for th in [2048, 4096, 8192, 16384, 32768, 65536]:
        n = int((arr > th).sum())
        print(f"  > {th:>6,}: {n:>6,}개 ({n/len(arr)*100:5.2f}%)")

    print(f"\n🔝 토큰 수 상위 10개")
    for rank, (n, idx, s) in enumerate(top_samples, 1):
        # messages에서 식별 가능한 정보 뽑기 (채널/영상명이 user content 앞쪽에 있음)
        user_msg = next((m["content"] for m in s["messages"] if m["role"] == "user"), "")
        head = user_msg[:120].replace("\n", " / ")
        print(f"  {rank:2d}. {n:>8,} tokens  idx={idx}  | {head}")

/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 토크나이저 로딩: unsloth/Qwen3.5-9B

📂 ./data/7_qwen_dataset_train.jsonl
  샘플 수: 63,080


토큰 계산: 100%|██████████| 63080/63080 [02:05<00:00, 503.89it/s] 



📊 전체 토큰 수 (prompt + completion)
  min   :       235
  mean  :     1,817
  median:     1,244
  p90   :     3,757
  p95   :     5,129
  p99   :     9,668
  p99.9 :    25,524
  max   :    93,513

📊 prompt / completion 분리
  prompt      max:     3,254  mean:       489
  completion  max:    92,949  mean:     1,327

📊 MAX_SEQ_LENGTH=8192 기준
  초과: 945개 (1.50%)
  >  2,048: 19,405개 (30.76%)
  >  4,096:  5,199개 ( 8.24%)
  >  8,192:    945개 ( 1.50%)
  > 16,384:    180개 ( 0.29%)
  > 32,768:     33개 ( 0.05%)
  > 65,536:      5개 ( 0.01%)

🔝 토큰 수 상위 10개
   1.   93,513 tokens  idx=48024  | 채널: 양띵 유튜브 / 영상: 근데 그럴만해 /  / [0.1-1.0] 보이세요? / [1.7-4.0] 저거 검은색 패션 아니고 / [4.0-6.4] 페인트 하다가 틴 거거든요? / [6.4-11.0] 자기가 봤을 때는 패션처럼 보일 테
   2.   87,258 tokens  idx=56264  | 채널: 찰스엔터 / 영상: 영지님한테 슈퍼챗 받다.. 가문의 영광_ /  / [0.1-2.0] 어? 이영진 님! / [2.0-3.5] 설마 내가 생각한 이영진 아니겠지? / [3.5-6.0] 혹시 래퍼 이영진 님이 계세요? / [6.0-
   3.   82,287 tokens  idx=38972  | 채널: 말왕TV / 영상: 클럽 프리패스상 /  / [0.1-2.0] 야 이거 뭐야? / [3.2-4.2] 여기 클럽이야 잉크리? / [7.6-8

토큰 계산: 100%|██████████| 3320/3320 [00:06<00:00, 490.95it/s]


📊 전체 토큰 수 (prompt + completion)
  min   :       243
  mean  :     1,827
  median:     1,234
  p90   :     3,835
  p95   :     5,188
  p99   :     9,952
  p99.9 :    24,915
  max   :    48,331

📊 prompt / completion 분리
  prompt      max:     2,188  mean:       493
  completion  max:    47,563  mean:     1,334

📊 MAX_SEQ_LENGTH=8192 기준
  초과: 56개 (1.69%)
  >  2,048:  1,021개 (30.75%)
  >  4,096:    286개 ( 8.61%)
  >  8,192:     56개 ( 1.69%)
  > 16,384:      8개 ( 0.24%)
  > 32,768:      2개 ( 0.06%)
  > 65,536:      0개 ( 0.00%)

🔝 토큰 수 상위 10개
   1.   48,331 tokens  idx=1868  | 채널: 대도서관TV (buzzbean11) / 영상: 소울리스좌 덕몽어스 버전 /  / [0.1-2.0] 가니는 오늘 뭘 보여줄건가요? / [2.0-6.8] 혹시 소울리스좌 덕목어스 버전으로 가니가 직접 계산을 해봤습니다 / [6.8
   2.   34,230 tokens  idx=2641  | 채널: 오빠두엑셀 l 엑셀 강의 대표채널 / 영상: 엑셀 행 추가, 이제 ＂Shift＂ 키로 편하게 작업하세요 (⚡써보면 정말 편리합니다) /  / [0.1-5.4] 엑셀에서 빈칸을 추가할 때 매번 마우스 우클릭으로 추가해서 
   3.   27,732 tokens  idx=2892  | 채널: 준우 / 영상: 전문대라고 개무시하던 친구 뭔가 이상한데 /  / [0.1-2.0] 존문대 나왔으면서 개나 대지 말고 그냥 한국에서 살라니까 / [2.0-3.4

In [8]:
# %% MAX_SEQ_LENGTH=16384 필터링 시 채널별 손실 확인
import json
import re
from collections import Counter, defaultdict
from tqdm import tqdm
from transformers import AutoTokenizer

# -------------------------
# Config
# -------------------------
MODEL_NAME = "unsloth/Qwen3.5-9B"
MAX_SEQ_LENGTH = 16384
DATASET_PATHS = [
    "./data/7_qwen_dataset_train.jsonl",
    "./data/7_qwen_dataset_eval.jsonl",
]

# -------------------------
# 토크나이저
# -------------------------
print(f"📦 토크나이저 로딩: {MODEL_NAME}")
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)


def extract_channel(user_content: str) -> str:
    """user 메시지에서 '채널: XXX' 추출"""
    m = re.search(r"채널:\s*(.+?)(?:\n|$)", user_content)
    return m.group(1).strip() if m else "<unknown>"


# -------------------------
# 필터링 & 채널 집계
# -------------------------
for path in DATASET_PATHS:
    print(f"\n{'='*60}")
    print(f"📂 {path}")
    print('='*60)

    with open(path, "r", encoding="utf-8") as f:
        samples = [json.loads(line) for line in f]

    # 채널별 전체 샘플 수
    total_per_channel = Counter()
    for s in samples:
        user_msg = next(m["content"] for m in s["messages"] if m["role"] == "user")
        total_per_channel[extract_channel(user_msg)] += 1

    # 초과 샘플 찾기
    cut_per_channel = Counter()
    cut_samples = []  # (channel, video_preview, n_tokens, idx)

    for i, sample in enumerate(tqdm(samples, desc="토큰 계산")):
        msgs = sample["messages"]
        prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
        asst_msgs = [m for m in msgs if m["role"] == "assistant"]

        prompt_text = tok.apply_chat_template(
            prompt_msgs, tokenize=False, add_generation_prompt=True
        )
        completion_text = asst_msgs[0]["content"] + "<|im_end|>\n" if asst_msgs else ""

        n_total = (
            len(tok.encode(prompt_text, add_special_tokens=False))
            + len(tok.encode(completion_text, add_special_tokens=False))
        )

        if n_total > MAX_SEQ_LENGTH:
            user_msg = next(m["content"] for m in msgs if m["role"] == "user")
            ch = extract_channel(user_msg)
            cut_per_channel[ch] += 1

            # 영상명도 같이 기록
            vm = re.search(r"영상:\s*(.+?)(?:\n|$)", user_msg)
            video = vm.group(1).strip()[:50] if vm else "?"
            cut_samples.append((ch, video, n_total, i))

    # -------------------------
    # 출력
    # -------------------------
    total_cut = sum(cut_per_channel.values())
    print(f"\n🚫 {MAX_SEQ_LENGTH} 초과: {total_cut:,}개 / {len(samples):,}개 "
          f"({total_cut/len(samples)*100:.2f}%)")
    print(f"   영향받은 채널 수: {len(cut_per_channel)}개 / 전체 {len(total_per_channel)}개")

    print(f"\n📊 채널별 손실 (손실 개수 / 해당 채널 전체) - 손실 많은 순")
    print(f"{'채널':<40} {'손실':>5} {'전체':>5} {'비율':>7}")
    print("-" * 65)
    for ch, cut_n in cut_per_channel.most_common():
        total_n = total_per_channel[ch]
        pct = cut_n / total_n * 100
        print(f"{ch[:38]:<40} {cut_n:>5} {total_n:>5} {pct:>6.1f}%")

    print(f"\n🔝 손실 샘플 상위 20개 (토큰 많은 순)")
    cut_samples.sort(key=lambda x: -x[2])
    print(f"{'채널':<25} {'영상':<40} {'tokens':>8} {'idx':>6}")
    print("-" * 85)
    for ch, video, n, idx in cut_samples:
        print(f"{ch[:23]:<25} {video[:38]:<40} {n:>8,} {idx:>6}")

📦 토크나이저 로딩: unsloth/Qwen3.5-9B

📂 ./data/7_qwen_dataset_train.jsonl


토큰 계산: 100%|██████████| 63080/63080 [02:25<00:00, 433.02it/s] 



🚫 16384 초과: 180개 / 63,080개 (0.29%)
   영향받은 채널 수: 45개 / 전체 664개

📊 채널별 손실 (손실 개수 / 해당 채널 전체) - 손실 많은 순
채널                                          손실    전체      비율
-----------------------------------------------------------------
LCK                                         23    95   24.2%
세바시 강연 Sebasi Talk                          16    95   16.8%
임다TV                                        15    95   15.8%
T1                                          12    95   12.6%
침착맨                                         10    95   10.5%
MBC every1                                   9    95    9.5%
승우아빠                                         8    95    8.4%
악어 유튜브                                       8    95    8.4%
말왕TV                                         7    95    7.4%
우왁굳                                          7    95    7.4%
준우                                           6    95    6.3%
최고다윽박                                        5    95    5.3%
밍모                                    

토큰 계산: 100%|██████████| 3320/3320 [00:07<00:00, 462.37it/s]


🚫 16384 초과: 8개 / 3,320개 (0.24%)
   영향받은 채널 수: 7개 / 전체 664개

📊 채널별 손실 (손실 개수 / 해당 채널 전체) - 손실 많은 순
채널                                          손실    전체      비율
-----------------------------------------------------------------
T1                                           2     5   40.0%
KBS HUMAN _ 뭉클티비                             1     5   20.0%
LCK                                          1     5   20.0%
대도서관TV (buzzbean11)                          1     5   20.0%
악어 유튜브                                       1     5   20.0%
오빠두엑셀 l 엑셀 강의 대표채널                           1     5   20.0%
준우                                           1     5   20.0%

🔝 손실 샘플 상위 20개 (토큰 많은 순)
채널                        영상                                         tokens    idx
-------------------------------------------------------------------------------------
대도서관TV (buzzbean11)       소울리스좌 덕몽어스 버전                              48,331   1868
오빠두엑셀 l 엑셀 강의 대표채널        엑셀 행 추가, 이제 ＂Shift＂ 키로 편하게 작업하세요 (⚡써보면     

In [5]:
# %% 손실 샘플이 채널 내에서 극단치인지 확인
import json
import re
import numpy as np
from collections import defaultdict
from tqdm import tqdm
from transformers import AutoTokenizer

MODEL_NAME = "unsloth/Qwen3.5-9B"
MAX_SEQ_LENGTH = 16384
PATH = "./data/7_qwen_dataset_train.jsonl"

# 확인하고 싶은 채널들 (손실 상위)
TARGET_CHANNELS = [
    "LCK", "세바시 강연 Sebasi Talk", "임다TV", "T1", "침착맨",
    "MBC every1", "승우아빠", "악어 유튜브", "말왕TV", "우왁굳",
    "양띵 유튜브", "찰스엔터",
]

print(f"📦 토크나이저 로딩: {MODEL_NAME}")
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)


def extract_channel(user_content: str) -> str:
    m = re.search(r"채널:\s*(.+?)(?:\n|$)", user_content)
    return m.group(1).strip() if m else "<unknown>"


# 채널별 전체 토큰 수 수집
channel_tokens = defaultdict(list)

with open(PATH, "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f]

for sample in tqdm(samples, desc="토큰 계산"):
    msgs = sample["messages"]
    user_msg = next(m["content"] for m in msgs if m["role"] == "user")
    ch = extract_channel(user_msg)

    if ch not in TARGET_CHANNELS:
        continue

    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]
    prompt_text = tok.apply_chat_template(
        prompt_msgs, tokenize=False, add_generation_prompt=True
    )
    completion_text = asst_msgs[0]["content"] + "<|im_end|>\n" if asst_msgs else ""
    n = (
        len(tok.encode(prompt_text, add_special_tokens=False))
        + len(tok.encode(completion_text, add_special_tokens=False))
    )
    channel_tokens[ch].append(n)

# 채널별 분포 출력
print(f"\n{'채널':<28} {'N':>4} {'min':>6} {'median':>7} {'p90':>7} {'p95':>7} {'p99':>7} {'max':>8} {'>16k':>5}")
print("-" * 95)
for ch in TARGET_CHANNELS:
    arr = np.array(channel_tokens[ch])
    if len(arr) == 0:
        continue
    over = int((arr > MAX_SEQ_LENGTH).sum())
    print(
        f"{ch[:26]:<28} "
        f"{len(arr):>4} "
        f"{int(arr.min()):>6,} "
        f"{int(np.median(arr)):>7,} "
        f"{int(np.percentile(arr, 90)):>7,} "
        f"{int(np.percentile(arr, 95)):>7,} "
        f"{int(np.percentile(arr, 99)):>7,} "
        f"{int(arr.max()):>8,} "
        f"{over:>5}"
    )

📦 토크나이저 로딩: unsloth/Qwen3.5-9B


토큰 계산: 100%|██████████| 63080/63080 [00:10<00:00, 6195.38it/s]  


채널                              N    min  median     p90     p95     p99      max  >16k
-----------------------------------------------------------------------------------------------
LCK                            95    430   9,702  23,649  26,570  35,027   54,786    23
세바시 강연 Sebasi Talk             95    934   2,732  45,506  56,181  67,344   68,551    16
임다TV                           95    854   8,870  18,970  21,708  32,327   55,386    15
T1                             95    437   4,400  18,531  22,154  46,560   47,052    12
침착맨                            95    352   5,339  15,750  18,137  25,171   29,921    10
MBC every1                     95    533   5,549  15,761  20,272  22,468   26,628     9
승우아빠                           95    966   2,268   4,609  25,939  28,947   30,323     8
악어 유튜브                         95  1,736   6,418  15,425  20,553  36,320   59,327     8
말왕TV                           95    264   1,511   9,504  19,614  51,186   82,287     7
우왁굳                    

In [9]:
# %% 생성된 JSON 토큰 분포 분석
import os
import json
import numpy as np
from collections import defaultdict
from tqdm import tqdm

OUTPUT_DIR = "./data/7_qwen_test"

# -------------------------
# JSON 파일 수집
# -------------------------
json_paths = []
for root, _, files in os.walk(OUTPUT_DIR):
    for fn in files:
        if fn.endswith(".json"):
            json_paths.append(os.path.join(root, fn))

print(f"📂 수집된 JSON: {len(json_paths):,}개")

# -------------------------
# 로드
# -------------------------
records = []
n_broken = 0
for p in tqdm(json_paths, desc="JSON 로드"):
    try:
        with open(p, "r", encoding="utf-8") as f:
            r = json.load(f)
        records.append({
            "orig_channel":      r.get("orig_channel"),
            "video_name":        r.get("video_name"),
            "cond_channel":      r.get("cond_channel"),
            "prompt_tokens":     r.get("prompt_tokens"),
            "completion_tokens": r.get("completion_tokens"),
            "n_instances":       len(r.get("instances", [])),
            "elapsed_sec":       r.get("elapsed_sec"),
        })
    except Exception:
        n_broken += 1

if n_broken:
    print(f"⚠️  파싱 실패: {n_broken}개")

print(f"✅ 유효: {len(records):,}개")


# -------------------------
# 토큰 분포
# -------------------------
def describe(name, arr):
    arr = np.array([x for x in arr if x is not None])
    if len(arr) == 0:
        print(f"{name}: (empty)")
        return
    print(f"\n📊 {name}  (N={len(arr):,})")
    print(f"  min   : {int(arr.min()):>9,}")
    print(f"  mean  : {arr.mean():>9,.0f}")
    print(f"  median: {int(np.median(arr)):>9,}")
    print(f"  p90   : {int(np.percentile(arr, 90)):>9,}")
    print(f"  p95   : {int(np.percentile(arr, 95)):>9,}")
    print(f"  p99   : {int(np.percentile(arr, 99)):>9,}")
    print(f"  max   : {int(arr.max()):>9,}")


prompts     = [r["prompt_tokens"]     for r in records]
completions = [r["completion_tokens"] for r in records]
totals      = [
    (r["prompt_tokens"] or 0) + (r["completion_tokens"] or 0)
    for r in records if r["prompt_tokens"] is not None and r["completion_tokens"] is not None
]

describe("prompt_tokens",     prompts)
describe("completion_tokens", completions)
describe("total (prompt+completion)", totals)


# -------------------------
# 임계값별 초과
# -------------------------
print(f"\n📊 completion_tokens 임계값별 초과")
c_arr = np.array([c for c in completions if c is not None])
for th in [1024, 2048, 4096, 8192, 16384, 32768, 65536, 126976]:
    n = int((c_arr > th).sum())
    print(f"  > {th:>6,}: {n:>5,}개 ({n/len(c_arr)*100:5.2f}%)")


# -------------------------
# 인스턴스 개수 분포
# -------------------------
describe("n_instances (생성된 텔롭 개수)", [r["n_instances"] for r in records])


# -------------------------
# completion_tokens가 MAX_TOKENS에 딱 붙은 샘플 (= 잘렸을 가능성)
# -------------------------
MAX_TOKENS = 126976
near_cap = [r for r in records if (r["completion_tokens"] or 0) >= MAX_TOKENS - 10]
print(f"\n⚠️  MAX_TOKENS({MAX_TOKENS}) 한계 도달 (잘렸을 가능성): {len(near_cap)}개")
for r in near_cap[:10]:
    print(f"  [{r['orig_channel']}] {r['video_name']}  "
          f"comp={r['completion_tokens']}  inst={r['n_instances']}")


# -------------------------
# completion 토큰 상위 20개
# -------------------------
records_sorted = sorted(
    [r for r in records if r["completion_tokens"] is not None],
    key=lambda x: -x["completion_tokens"]
)
print(f"\n🔝 completion_tokens 상위 20개")
print(f"{'채널':<28} {'영상':<40} {'prompt':>7} {'comp':>8} {'inst':>5} {'sec':>6}")
print("-" * 100)
for r in records_sorted[:20]:
    print(
        f"{(r['orig_channel'] or '')[:26]:<28} "
        f"{(r['video_name'] or '')[:38]:<40} "
        f"{(r['prompt_tokens'] or 0):>7,} "
        f"{(r['completion_tokens'] or 0):>8,} "
        f"{r['n_instances']:>5} "
        f"{(r['elapsed_sec'] or 0):>6.1f}"
    )


# -------------------------
# 채널별 평균 completion_tokens (상위 20)
# -------------------------
by_channel = defaultdict(list)
for r in records:
    if r["completion_tokens"] is not None:
        by_channel[r["orig_channel"]].append(r["completion_tokens"])

ch_stats = [
    (ch, len(vs), int(np.mean(vs)), int(np.median(vs)), int(max(vs)))
    for ch, vs in by_channel.items()
]
ch_stats.sort(key=lambda x: -x[2])  # 평균 내림차순

print(f"\n📊 채널별 completion_tokens 평균 상위 20")
print(f"{'채널':<32} {'N':>4} {'mean':>7} {'median':>7} {'max':>8}")
print("-" * 68)
for ch, n, mean, med, mx in ch_stats[:20]:
    print(f"{(ch or '')[:30]:<32} {n:>4} {mean:>7,} {med:>7,} {mx:>8,}")

📂 수집된 JSON: 3,038개


JSON 로드: 100%|██████████| 3038/3038 [00:00<00:00, 3564.44it/s]

✅ 유효: 3,038개

📊 prompt_tokens  (N=3,038)
  min   :       230
  mean  :       495
  median:       438
  p90   :       794
  p95   :       907
  p99   :     1,285
  max   :     2,188

📊 completion_tokens  (N=3,038)
  min   :         5
  mean  :     1,874
  median:     1,123
  p90   :     4,277
  p95   :     5,866
  p99   :     9,708
  max   :    18,940

📊 total (prompt+completion)  (N=3,038)
  min   :       238
  mean  :     2,368
  median:     1,744
  p90   :     4,873
  p95   :     6,434
  p99   :    10,345
  max   :    19,816

📊 completion_tokens 임계값별 초과
  >  1,024: 1,632개 (53.72%)
  >  2,048:   998개 (32.85%)
  >  4,096:   351개 (11.55%)
  >  8,192:    68개 ( 2.24%)
  > 16,384:     1개 ( 0.03%)
  > 32,768:     0개 ( 0.00%)
  > 65,536:     0개 ( 0.00%)
  > 126,976:     0개 ( 0.00%)

📊 n_instances (생성된 텔롭 개수)  (N=3,038)
  min   :         0
  mean  :        85
  median:        49
  p90   :       191
  p95   :       264
  p99   :       444
  max   :       859

⚠️  MAX_TOKENS(126976) 한계 도달 (잘렸을 

In [10]:
# %% 가장 최근 생성된 JSON의 토큰 수
import os
import json
import glob

OUTPUT_DIR = "./data/7_qwen_test"

# 모든 json 수집 후 mtime 최신순
json_paths = glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True)
json_paths.sort(key=lambda p: os.path.getmtime(p), reverse=True)

N = 10  # 최근 N개
print(f"📂 전체 JSON: {len(json_paths):,}개, 최근 {N}개 표시\n")

print(f"{'mtime':<20} {'채널':<22} {'영상':<32} {'prompt':>7} {'comp':>7} {'inst':>5}")
print("-" * 100)

for p in json_paths[:N]:
    import datetime
    mtime = datetime.datetime.fromtimestamp(os.path.getmtime(p)).strftime("%Y-%m-%d %H:%M:%S")
    try:
        with open(p, "r", encoding="utf-8") as f:
            r = json.load(f)
        ch   = (r.get("orig_channel") or "")[:20]
        vid  = (r.get("video_name") or "")[:30]
        pt   = r.get("prompt_tokens") or 0
        ct   = r.get("completion_tokens") or 0
        nins = len(r.get("instances", []))
        print(f"{mtime:<20} {ch:<22} {vid:<32} {pt:>7,} {ct:>7,} {nins:>5}")
    except Exception as e:
        print(f"{mtime:<20} [ERROR] {p}: {e}")
        

📂 전체 JSON: 3,038개, 최근 10개 표시

mtime                채널                     영상                                prompt    comp  inst
----------------------------------------------------------------------------------------------------
2026-04-21 14:18:57  AKMU                   이찬혁 (LEE CHANHYUK) - '1조 (1 TR       347   2,922   134
2026-04-21 14:01:52  한문철 TV                 #2727. 전동 킥보드가 왜 저기서ㅠ #shorts        379   7,115   191
2026-04-21 14:01:45  티티팡팡 TTPP              [#노는브로] 야구선수 유전자가 강력하다더니... #S       622  14,030   518
2026-04-21 14:00:56  헬로라이프                  ㅋㅋㅋ겁나 당돌했던 초딩장윤정ㅋㅋㅋㅋ 장윤정 #이찬원        768   1,543    70
2026-04-21 14:00:35  현대자동차그룹(HYUNDAI)       EV3로 즐기는 여유로운 전기차 라이프 ｜ #Short       246     116     5
2026-04-21 14:00:35  헤이지니 Hey Jini          어떤 숟가락이 가장 좋을까？ 지니랑 테스트해보자! #k       343   1,189    62
2026-04-21 14:00:25  헨리 HENRY LAU           헨리와 스탭의 찐텐🤣🤣                         318     362    17
2026-04-21 13:56:34  헨리 HENRY LAU           It’s You..🫶🏻 #henry              

In [11]:
# %% 진단: GT vs 생성 인스턴스 분포 + 반복 탐지
import os
import re
import json
import glob
import numpy as np
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test"

# -------------------------
# 1) GT 로딩 (eval.jsonl의 assistant turn)
# -------------------------
def parse_user_text(c):
    if isinstance(c, str): return c
    if isinstance(c, list):
        return "\n".join(p.get("text", "") for p in c if isinstance(p, dict))
    return ""

gt_records = {}  # (channel, video) -> {"instances": [...], "prompt_len_chars": int, "num_stt": int}
with open(EVAL_JSONL, encoding="utf-8") as f:
    for line in tqdm(f, desc="eval GT 로딩"):
        ex = json.loads(line)
        sys_msg = next(m for m in ex["messages"] if m["role"] == "system")
        usr_msg = next(m for m in ex["messages"] if m["role"] == "user")
        asst    = next(m for m in ex["messages"] if m["role"] == "assistant")
        user_text = parse_user_text(usr_msg["content"])

        m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
        m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
        if not m_ch or not m_vn: continue
        ch, vn = m_ch.group(1).strip(), m_vn.group(1).strip()

        try:
            gt = json.loads(asst["content"])
            insts = gt.get("instances", [])
        except Exception:
            insts = []

        n_stt = len([l for l in user_text.splitlines() if re.match(r"\[\d", l)])

        gt_records[(ch, vn)] = {
            "instances":       insts,
            "n_inst_gt":       len(insts),
            "total_chars_gt":  sum(len(i.get("text", "")) for i in insts),
            "dur_gt":          sum(max(0.0, i.get("end_sec", 0) - i.get("start_sec", 0)) for i in insts),
            "max_end_gt":      max((i.get("end_sec", 0) for i in insts), default=0.0),
            "n_stt":           n_stt,
            "prompt_chars":    len(user_text),
        }

print(f"GT: {len(gt_records)}개 영상")


# -------------------------
# 2) 생성 결과 로딩 (본채널 조건만)
# -------------------------
gen_records = {}  # same key
gen_paths = glob.glob(os.path.join(OUTPUT_DIR, "*", "*", "*.json"))
print(f"생성 결과 파일: {len(gen_paths)}개")

for p in tqdm(gen_paths, desc="생성 로딩"):
    try:
        with open(p, encoding="utf-8") as f:
            r = json.load(f)
    except Exception:
        continue
    orig = r.get("orig_channel")
    vid  = r.get("video_name")
    cond = r.get("cond_channel")
    # 본채널 조건만 (swap 제외)
    if orig != cond:
        continue

    # 영상명 포맷은 eval.jsonl에 strip_video_id 적용된 상태임에 유의
    # strip: "xxx__<11char>" 의 "__" 뒷부분 제거
    # 따라서 gen_records의 vid는 이미 stripped로 들어있어야 함
    insts = r.get("instances", [])
    if not isinstance(insts, list):
        insts = []

    gen_records[(orig, vid)] = {
        "instances":       insts,
        "n_inst_gen":      len(insts),
        "total_chars_gen": sum(len(i.get("text", "")) if isinstance(i, dict) else 0 for i in insts),
        "dur_gen":         sum(max(0.0, i.get("end_sec", 0) - i.get("start_sec", 0))
                               for i in insts if isinstance(i, dict)),
        "max_end_gen":     max((i.get("end_sec", 0) for i in insts if isinstance(i, dict)), default=0.0),
        "elapsed":         r.get("elapsed_sec"),
        "prompt_tokens":   r.get("prompt_tokens"),
        "completion_tokens": r.get("completion_tokens"),
        "raw_output":      r.get("raw_output", ""),
    }

print(f"GEN (본채널): {len(gen_records)}개")


# -------------------------
# 3) 매칭 + DataFrame
# -------------------------
# eval.jsonl의 video는 strip_video_id 적용된 값 (video_name.rsplit("__", 1)[0])
# 7_qwen_test.ipynb는 re.search(r"영상:\s*([^\n]+)")로 그 stripped 값을 뽑아서 그대로 파일명에 쓴다 → 일치
keys = sorted(set(gt_records.keys()) & set(gen_records.keys()))
print(f"매칭: {len(keys)}개")

rows = []
for k in keys:
    g = gt_records[k]
    p = gen_records[k]
    rows.append({
        "channel":         k[0],
        "video":           k[1],
        "n_stt":           g["n_stt"],
        "prompt_chars":    g["prompt_chars"],
        "n_gt":            g["n_inst_gt"],
        "n_gen":           p["n_inst_gen"],
        "ratio":           p["n_inst_gen"] / max(1, g["n_inst_gt"]),
        "chars_gt":        g["total_chars_gt"],
        "chars_gen":       p["total_chars_gen"],
        "dur_gt":          g["dur_gt"],
        "dur_gen":         p["dur_gen"],
        "max_end_gt":      g["max_end_gt"],
        "max_end_gen":     p["max_end_gen"],
        "completion_tok":  p["completion_tokens"],
        "elapsed":         p["elapsed"],
    })
df = pd.DataFrame(rows)

print("\n=== 전체 분포 ===")
print(df[["n_gt", "n_gen", "ratio", "completion_tok", "elapsed"]].describe(
    percentiles=[.1, .25, .5, .75, .9, .95, .99]
))

# GT 0인 영상 (텔롭 없음) 별도 확인
df_empty_gt = df[df["n_gt"] == 0]
print(f"\n=== GT가 0개인 영상 ({len(df_empty_gt)}개) ===")
if len(df_empty_gt):
    print(df_empty_gt["n_gen"].describe())
    print(f"  GT=0인데 모델이 생성한 인스턴스: "
          f"{(df_empty_gt['n_gen'] > 0).sum()}개 영상, "
          f"총 {df_empty_gt['n_gen'].sum()}개 instance")

# 배수로 과생성하는 비율
print(f"\n=== 과생성 지표 (GT>0인 영상만, n={len(df[df['n_gt']>0])}) ===")
df_pos = df[df["n_gt"] > 0]
for thr in [1.5, 2.0, 3.0, 5.0, 10.0]:
    n = (df_pos["ratio"] >= thr).sum()
    print(f"  ratio >= {thr:.1f}: {n}개 ({n/len(df_pos)*100:.1f}%)")

print(f"\n  평균 ratio: {df_pos['ratio'].mean():.2f}")
print(f"  중앙값 ratio: {df_pos['ratio'].median():.2f}")


# -------------------------
# 4) max_tokens에 걸린 (EOS 못 낸) 샘플 추정
# -------------------------
# completion_tokens가 max_tokens에 근접한 경우 = 생성이 잘린 경우
MAX_TOKENS = 126976
df_stuck = df[df["completion_tok"] >= MAX_TOKENS - 100]
print(f"\n=== max_tokens 한계 도달 (completion_tok >= MAX-100) ===")
print(f"  {len(df_stuck)}개 ({len(df_stuck)/len(df)*100:.2f}%)")
if len(df_stuck):
    print(df_stuck[["channel", "video", "n_stt", "n_gt", "n_gen", "completion_tok"]].head(10))


# -------------------------
# 5) 반복(중복 instance) 탐지
# -------------------------
def count_duplicates(insts):
    """instances 내 중복 (text, start_sec, end_sec) 카운트"""
    if not insts: return 0, 0
    keys = []
    for i in insts:
        if not isinstance(i, dict): continue
        keys.append((i.get("text"), round(float(i.get("start_sec", 0)), 2),
                     round(float(i.get("end_sec", 0)), 2)))
    c = Counter(keys)
    exact_dup = sum(v - 1 for v in c.values() if v > 1)
    # text만 중복
    text_only = Counter(k[0] for k in keys)
    text_dup = sum(v - 1 for v in text_only.values() if v > 1)
    return exact_dup, text_dup

dup_stats = []
for k in keys:
    insts = gen_records[k]["instances"]
    ed, td = count_duplicates(insts)
    dup_stats.append({"channel": k[0], "video": k[1], "n_gen": len(insts),
                      "exact_dup": ed, "text_dup": td})
dup_df = pd.DataFrame(dup_stats)
print(f"\n=== 반복 탐지 ===")
print(f"  exact 중복 있는 영상: {(dup_df['exact_dup'] > 0).sum()}개")
print(f"  text 중복 있는 영상:  {(dup_df['text_dup']  > 0).sum()}개")
print(f"  평균 text 중복 수(중복 있는 것만): "
      f"{dup_df[dup_df['text_dup']>0]['text_dup'].mean():.1f}")
print("\n  text 중복 top 10:")
print(dup_df.sort_values("text_dup", ascending=False).head(10))


# -------------------------
# 6) STT 길이 vs 과생성 상관
# -------------------------
print("\n=== STT 길이 bin별 평균 ratio (긴 입력에서 과생성이 심한가?) ===")
df["stt_bin"] = pd.qcut(df["n_stt"], q=5, duplicates="drop")
agg = df[df["n_gt"] > 0].groupby("stt_bin", observed=True).agg(
    n=("ratio", "size"),
    mean_ratio=("ratio", "mean"),
    median_ratio=("ratio", "median"),
    mean_gt=("n_gt", "mean"),
    mean_gen=("n_gen", "mean"),
    mean_comp_tok=("completion_tok", "mean"),
)
print(agg)


# -------------------------
# 7) 가장 과생성된 영상 몇 개의 raw_output 앞/뒤 보기
# -------------------------
print("\n=== ratio 상위 5개 영상의 raw_output 샘플 ===")
top = df[df["n_gt"] > 0].nlargest(5, "ratio")
for _, r in top.iterrows():
    k = (r["channel"], r["video"])
    raw = gen_records[k]["raw_output"]
    print(f"\n--- {r['channel']} / {r['video']} ---")
    print(f"  n_gt={r['n_gt']}, n_gen={r['n_gen']}, ratio={r['ratio']:.1f}, "
          f"completion_tok={r['completion_tok']}")
    print(f"  raw 앞 300자: {raw[:300]}")
    print(f"  raw 뒤 300자: {raw[-300:]}")

eval GT 로딩: 3320it [00:00, 12086.09it/s]


GT: 3315개 영상
생성 결과 파일: 3038개


생성 로딩: 100%|██████████| 3038/3038 [00:00<00:00, 4401.67it/s]


GEN (본채널): 3038개
매칭: 3038개

=== 전체 분포 ===
              n_gt        n_gen        ratio  completion_tok      elapsed
count  3038.000000  3038.000000  3038.000000     3038.000000  3038.000000
mean     54.086899    84.667544     9.254429     1873.600066    97.915553
std      87.422765    96.509149    27.529957     2123.100426   172.092944
min       0.000000     0.000000     0.000000        5.000000     0.130000
10%       1.000000     1.000000     0.073769       25.000000     0.747000
25%       5.000000    17.000000     0.500000      397.000000    13.035000
50%      32.000000    49.000000     1.028219     1123.000000    35.380000
75%      70.000000   131.000000     4.000000     2827.000000   101.180000
90%     126.000000   191.300000    20.260000     4277.600000   297.454000
95%     175.150000   264.150000    54.000000     5866.900000   426.389500
99%     352.000000   444.630000   157.420000     9708.170000   760.297700
max    1887.000000   859.000000   374.000000    18940.000000  1486.990

In [7]:
# %% 토큰 길이 분포 비교: GT vs 생성된 JSON
import os
import json
import glob
import re
import numpy as np
from transformers import AutoTokenizer
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test_01"
MODEL_PATH = "./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"

# -------------------------
# 1) tokenizer 로드
# -------------------------
print("🔧 tokenizer 로딩...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)


def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


def parse_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(part.get("text", "") for part in msg_content if isinstance(part, dict))
    return ""


# -------------------------
# 2) eval.jsonl → GT 수집
# -------------------------
print("\n📖 eval.jsonl 파싱 + GT 토큰 수 계산...")
gt_map = {}  # (channel, video) → gt_tokens

with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in tqdm(lines, desc="GT 수집"):
    ex = json.loads(line)
    msgs = ex["messages"]

    user_msg = next(m for m in msgs if m["role"] == "user")
    asst_msg = next((m for m in msgs if m["role"] == "assistant"), None)
    if asst_msg is None:
        continue

    user_text = parse_user_text(user_msg["content"])
    asst_text = asst_msg["content"] if isinstance(asst_msg["content"], str) \
                else parse_user_text(asst_msg["content"])

    m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
    m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
    if not m_ch or not m_vn:
        continue

    gt_map[(m_ch.group(1).strip(), m_vn.group(1).strip())] = count_tokens(asst_text)

print(f"✅ GT 항목 수: {len(gt_map)}")


# -------------------------
# 3) 생성된 JSON 수집 (본채널 조건만)
# -------------------------
print("\n📦 생성된 JSON 읽기...")
json_files = glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True)

pairs = []
missing_gt = 0
missing_completion = 0

for path in tqdm(json_files, desc="생성 JSON 파싱"):
    try:
        with open(path, "r", encoding="utf-8") as f:
            rec = json.load(f)
    except Exception:
        continue

    orig_ch = rec.get("orig_channel")
    cond_ch = rec.get("cond_channel")
    video   = rec.get("video_name")

    if orig_ch != cond_ch:
        continue

    key = (orig_ch, video)
    if key not in gt_map:
        missing_gt += 1
        continue

    pred_tok = rec.get("completion_tokens")
    if pred_tok is None:
        raw = rec.get("raw_output", "")
        if not raw:
            missing_completion += 1
            continue
        pred_tok = count_tokens(raw)

    pairs.append({
        "channel":  orig_ch,
        "video":    video,
        "gt_tok":   gt_map[key],
        "pred_tok": pred_tok,
    })

print(f"✅ 매칭된 쌍: {len(pairs)}")
if missing_gt:
    print(f"⚠️ GT 없는 생성물: {missing_gt}")
if missing_completion:
    print(f"⚠️ completion_tokens/raw_output 없음: {missing_completion}")


# -------------------------
# 4) 통계 출력
# -------------------------
gt_lens   = np.array([p["gt_tok"]   for p in pairs])
pred_lens = np.array([p["pred_tok"] for p in pairs])

def print_stats(arr, name):
    print(f"\n📊 {name}")
    print(f"  개수:    {len(arr)}")
    print(f"  mean:    {arr.mean():.1f}")
    print(f"  median:  {np.median(arr):.1f}")
    print(f"  std:     {arr.std():.1f}")
    print(f"  min:     {arr.min()}")
    print(f"  max:     {arr.max()}")
    print(f"  p25:     {np.percentile(arr, 25):.1f}")
    print(f"  p75:     {np.percentile(arr, 75):.1f}")
    print(f"  p95:     {np.percentile(arr, 95):.1f}")
    print(f"  p99:     {np.percentile(arr, 99):.1f}")

print_stats(gt_lens,   "GT")
print_stats(pred_lens, "Pred")


# -------------------------
# 5) 나란히 비교 테이블
# -------------------------
print("\n📋 GT vs Pred 비교")
print(f"  {'metric':<10} {'GT':>12} {'Pred':>12} {'diff':>12}")
print(f"  {'-'*10} {'-'*12} {'-'*12} {'-'*12}")
for label, fn in [
    ("mean",   np.mean),
    ("median", np.median),
    ("std",    np.std),
    ("min",    np.min),
    ("max",    np.max),
    ("p25",    lambda a: np.percentile(a, 25)),
    ("p75",    lambda a: np.percentile(a, 75)),
    ("p95",    lambda a: np.percentile(a, 95)),
    ("p99",    lambda a: np.percentile(a, 99)),
]:
    g, p = fn(gt_lens), fn(pred_lens)
    print(f"  {label:<10} {g:>12.1f} {p:>12.1f} {p-g:>+12.1f}")


# -------------------------
# 6) 길이 비율 통계
# -------------------------
ratio = pred_lens / np.maximum(gt_lens, 1)
print(f"\n📏 길이 비율 (Pred / GT)")
print(f"  mean:   {ratio.mean():.3f}")
print(f"  median: {np.median(ratio):.3f}")
print(f"  p25:    {np.percentile(ratio, 25):.3f}")
print(f"  p75:    {np.percentile(ratio, 75):.3f}")
print(f"  < 0.5:  {(ratio < 0.5).sum():>6} ({100*(ratio < 0.5).mean():.1f}%)  ← Pred이 너무 짧음")
print(f"  < 0.8:  {(ratio < 0.8).sum():>6} ({100*(ratio < 0.8).mean():.1f}%)")
print(f"  0.8~1.2:{((ratio >= 0.8) & (ratio <= 1.2)).sum():>6} ({100*((ratio >= 0.8) & (ratio <= 1.2)).mean():.1f}%)  ← 유사한 길이")
print(f"  > 1.2:  {(ratio > 1.2).sum():>6} ({100*(ratio > 1.2).mean():.1f}%)")
print(f"  > 2.0:  {(ratio > 2.0).sum():>6} ({100*(ratio > 2.0).mean():.1f}%)  ← Pred이 너무 김")


# -------------------------
# 7) 텍스트 기반 히스토그램 (구간별 개수)
# -------------------------
print("\n📈 구간별 분포 (토큰 수)")
max_v = max(gt_lens.max(), pred_lens.max())
bins = np.linspace(0, np.percentile(np.concatenate([gt_lens, pred_lens]), 99), 11)
print(f"  {'range':<20} {'GT':>8} {'Pred':>8}")
print(f"  {'-'*20} {'-'*8} {'-'*8}")
for i in range(len(bins) - 1):
    lo, hi = int(bins[i]), int(bins[i+1])
    g_count = ((gt_lens   >= lo) & (gt_lens   < hi)).sum()
    p_count = ((pred_lens >= lo) & (pred_lens < hi)).sum()
    print(f"  {lo:>6} ~ {hi:<10} {g_count:>8} {p_count:>8}")

# p99 넘는 outlier
g_tail = (gt_lens   >= bins[-1]).sum()
p_tail = (pred_lens >= bins[-1]).sum()
print(f"  {int(bins[-1]):>6} +  {'':<10} {g_tail:>8} {p_tail:>8}")

🔧 tokenizer 로딩...


The tokenizer you are loading from './model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



📖 eval.jsonl 파싱 + GT 토큰 수 계산...


GT 수집: 100%|██████████| 3320/3320 [00:05<00:00, 563.09it/s]


✅ GT 항목 수: 3315

📦 생성된 JSON 읽기...


생성 JSON 파싱: 100%|██████████| 3038/3038 [00:00<00:00, 5005.82it/s]

✅ 매칭된 쌍: 3038

📊 GT
  개수:    3038
  mean:    1262.9
  median:  737.5
  std:     2088.9
  min:     4
  max:     47561
  p25:     107.0
  p75:     1640.0
  p95:     4096.4
  p99:     8537.1

📊 Pred
  개수:    3038
  mean:    1873.6
  median:  1123.0
  std:     2122.8
  min:     5
  max:     18940
  p25:     397.0
  p75:     2827.0
  p95:     5866.9
  p99:     9708.2

📋 GT vs Pred 비교
  metric               GT         Pred         diff
  ---------- ------------ ------------ ------------
  mean             1262.9       1873.6       +610.7
  median            737.5       1123.0       +385.5
  std              2088.9       2122.8        +33.9
  min                 4.0          5.0         +1.0
  max             47561.0      18940.0     -28621.0
  p25               107.0        397.0       +290.0
  p75              1640.0       2827.0      +1187.0
  p95              4096.4       5866.9      +1770.6
  p99              8537.1       9708.2      +1171.1

📏 길이 비율 (Pred / GT)
  mean:   20.311
  median

In [8]:
# %% 토큰 길이 분포 비교: GT vs 생성된 JSON (페어 기반)
import os
import json
import glob
import re
import numpy as np
from transformers import AutoTokenizer
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test_01"
MODEL_PATH = "./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"

# -------------------------
# 1) tokenizer 로드
# -------------------------
print("🔧 tokenizer 로딩...")
MODEL_PATH_ABS = os.path.abspath(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH_ABS)

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


def parse_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(part.get("text", "") for part in msg_content if isinstance(part, dict))
    return ""


# -------------------------
# 2) 먼저 완료된 JSON 파일 목록 수집
# -------------------------
print("\n📦 완료된 생성 JSON 수집...")
json_files = glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True)

done_map = {}  # (channel, video) → {pred_tok}
for path in tqdm(json_files, desc="생성 JSON 파싱"):
    try:
        with open(path, "r", encoding="utf-8") as f:
            rec = json.load(f)
    except Exception:
        continue

    orig_ch = rec.get("orig_channel")
    cond_ch = rec.get("cond_channel")
    video   = rec.get("video_name")

    if orig_ch != cond_ch:
        continue

    pred_tok = rec.get("completion_tokens")
    if pred_tok is None:
        raw = rec.get("raw_output", "")
        if not raw:
            continue
        pred_tok = count_tokens(raw)

    done_map[(orig_ch, video)] = pred_tok

print(f"✅ 완료된 생성물: {len(done_map)}")


# -------------------------
# 3) eval.jsonl 중 "완료된 것에 해당하는 GT만" 토큰화
# -------------------------
print("\n📖 완료된 샘플에 대응하는 GT만 계산...")
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    lines = f.readlines()

pairs = []
missing_in_eval = 0

for line in tqdm(lines, desc="GT 계산"):
    ex = json.loads(line)
    msgs = ex["messages"]

    user_msg = next(m for m in msgs if m["role"] == "user")
    asst_msg = next((m for m in msgs if m["role"] == "assistant"), None)
    if asst_msg is None:
        continue

    user_text = parse_user_text(user_msg["content"])
    m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
    m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
    if not m_ch or not m_vn:
        continue

    key = (m_ch.group(1).strip(), m_vn.group(1).strip())
    if key not in done_map:
        continue  # 아직 생성 안 된 것은 스킵

    asst_text = asst_msg["content"] if isinstance(asst_msg["content"], str) \
                else parse_user_text(asst_msg["content"])

    pairs.append({
        "channel":  key[0],
        "video":    key[1],
        "gt_tok":   count_tokens(asst_text),
        "pred_tok": done_map[key],
    })

# 완료됐는데 eval에 대응이 없는 경우 (거의 없을 것)
eval_keys = {p["channel"] + "|" + p["video"] for p in pairs}
for (ch, vd) in done_map:
    if ch + "|" + vd not in eval_keys:
        missing_in_eval += 1

print(f"✅ 매칭된 페어: {len(pairs)}")
if missing_in_eval:
    print(f"⚠️ eval에서 GT 못 찾은 완료물: {missing_in_eval}")


# -------------------------
# 4) 분포 통계
# -------------------------
gt_lens   = np.array([p["gt_tok"]   for p in pairs])
pred_lens = np.array([p["pred_tok"] for p in pairs])

def print_stats(arr, name):
    print(f"\n📊 {name}")
    print(f"  개수:    {len(arr)}")
    print(f"  mean:    {arr.mean():.1f}")
    print(f"  median:  {np.median(arr):.1f}")
    print(f"  std:     {arr.std():.1f}")
    print(f"  min:     {arr.min()}")
    print(f"  max:     {arr.max()}")
    print(f"  p25:     {np.percentile(arr, 25):.1f}")
    print(f"  p75:     {np.percentile(arr, 75):.1f}")
    print(f"  p95:     {np.percentile(arr, 95):.1f}")
    print(f"  p99:     {np.percentile(arr, 99):.1f}")

print_stats(gt_lens,   "GT (완료된 것만)")
print_stats(pred_lens, "Pred")


# -------------------------
# 5) 나란히 비교 테이블
# -------------------------
print("\n📋 GT vs Pred 비교 (같은 샘플끼리)")
print(f"  {'metric':<10} {'GT':>12} {'Pred':>12} {'diff':>12}")
print(f"  {'-'*10} {'-'*12} {'-'*12} {'-'*12}")
for label, fn in [
    ("mean",   np.mean),
    ("median", np.median),
    ("std",    np.std),
    ("min",    np.min),
    ("max",    np.max),
    ("p25",    lambda a: np.percentile(a, 25)),
    ("p75",    lambda a: np.percentile(a, 75)),
    ("p95",    lambda a: np.percentile(a, 95)),
    ("p99",    lambda a: np.percentile(a, 99)),
]:
    g, p = fn(gt_lens), fn(pred_lens)
    print(f"  {label:<10} {g:>12.1f} {p:>12.1f} {p-g:>+12.1f}")


# -------------------------
# 6) 페어별 차이 통계 (같은 샘플끼리)
# -------------------------
diff = pred_lens - gt_lens  # 양수면 Pred가 김
print("\n📐 샘플별 차이 (Pred - GT)")
print(f"  mean:   {diff.mean():+.1f}")
print(f"  median: {np.median(diff):+.1f}")
print(f"  std:    {diff.std():.1f}")
print(f"  min:    {diff.min():+.0f}")
print(f"  max:    {diff.max():+.0f}")
print(f"  Pred가 더 긴 샘플: {(diff > 0).sum()} ({100*(diff > 0).mean():.1f}%)")
print(f"  Pred가 더 짧은 샘플: {(diff < 0).sum()} ({100*(diff < 0).mean():.1f}%)")
print(f"  완전 동일: {(diff == 0).sum()}")


# -------------------------
# 7) 길이 비율 통계
# -------------------------
ratio = pred_lens / np.maximum(gt_lens, 1)
print(f"\n📏 길이 비율 (Pred / GT)")
print(f"  median: {np.median(ratio):.3f}")  # mean은 왜곡 심해서 제외
print(f"  p25:    {np.percentile(ratio, 25):.3f}")
print(f"  p75:    {np.percentile(ratio, 75):.3f}")
print(f"  < 0.5:  {(ratio < 0.5).sum():>6} ({100*(ratio < 0.5).mean():.1f}%)  ← Pred이 너무 짧음")
print(f"  < 0.8:  {(ratio < 0.8).sum():>6} ({100*(ratio < 0.8).mean():.1f}%)")
print(f"  0.8~1.2:{((ratio >= 0.8) & (ratio <= 1.2)).sum():>6} ({100*((ratio >= 0.8) & (ratio <= 1.2)).mean():.1f}%)  ← 유사한 길이")
print(f"  > 1.2:  {(ratio > 1.2).sum():>6} ({100*(ratio > 1.2).mean():.1f}%)")
print(f"  > 2.0:  {(ratio > 2.0).sum():>6} ({100*(ratio > 2.0).mean():.1f}%)  ← Pred이 너무 김")


# -------------------------
# 8) GT 길이 구간별로 Pred 편향 분석
# -------------------------
print("\n📊 GT 길이 구간별 Pred 평균 길이 (같은 bucket 내 비교)")
print(f"  {'GT range':<20} {'n':>6} {'GT mean':>10} {'Pred mean':>10} {'ratio':>8}")
print(f"  {'-'*20} {'-'*6} {'-'*10} {'-'*10} {'-'*8}")

gt_bins = [0, 100, 300, 700, 1500, 3000, 6000, 15000, 100000]
for i in range(len(gt_bins) - 1):
    lo, hi = gt_bins[i], gt_bins[i+1]
    mask = (gt_lens >= lo) & (gt_lens < hi)
    if mask.sum() == 0:
        continue
    g_m = gt_lens[mask].mean()
    p_m = pred_lens[mask].mean()
    print(f"  {lo:>6} ~ {hi:<10} {mask.sum():>6} {g_m:>10.0f} {p_m:>10.0f} {p_m/g_m:>8.2f}")

🔧 tokenizer 로딩...


The tokenizer you are loading from '/home/taeyoung/nfs-mount/chi2027/model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



📦 완료된 생성 JSON 수집...


생성 JSON 파싱: 100%|██████████| 3038/3038 [00:00<00:00, 5588.27it/s]


✅ 완료된 생성물: 3038

📖 완료된 샘플에 대응하는 GT만 계산...


GT 계산: 100%|██████████| 3320/3320 [00:04<00:00, 727.31it/s] 

✅ 매칭된 페어: 3043

📊 GT (완료된 것만)
  개수:    3043
  mean:    1261.4
  median:  736.0
  std:     2087.5
  min:     4
  max:     47561
  p25:     107.0
  p75:     1633.0
  p95:     4094.5
  p99:     8535.9

📊 Pred
  개수:    3043
  mean:    1876.0
  median:  1124.0
  std:     2123.4
  min:     5
  max:     18940
  p25:     397.0
  p75:     2832.5
  p95:     5871.4
  p99:     9700.2

📋 GT vs Pred 비교 (같은 샘플끼리)
  metric               GT         Pred         diff
  ---------- ------------ ------------ ------------
  mean             1261.4       1876.0       +614.6
  median            736.0       1124.0       +388.0
  std              2087.5       2123.4        +35.9
  min                 4.0          5.0         +1.0
  max             47561.0      18940.0     -28621.0
  p25               107.0        397.0       +290.0
  p75              1633.0       2832.5      +1199.5
  p95              4094.5       5871.4      +1776.9
  p99              8535.9       9700.2      +1164.3

📐 샘플별 차이 (Pred - GT)
  me

In [9]:
# %% 과생성 심한 샘플 raw_output 확인
import os
import json
import glob
import re
import numpy as np
from transformers import AutoTokenizer
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test_01"
MODEL_PATH = "./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"

TOP_K = 10       # 상위 몇 개 볼지
TAIL_CHARS = 600 # 꼬리 몇 자 볼지
HEAD_CHARS = 300 # 머리 몇 자 볼지

# -------------------------
# 1) tokenizer + GT 로드
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(os.path.abspath(MODEL_PATH))

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

def parse_user_text(c):
    if isinstance(c, str): return c
    if isinstance(c, list):
        return "\n".join(p.get("text", "") for p in c if isinstance(p, dict))
    return ""

gt_map = {}  # (ch, vd) → gt_text
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="GT 로드"):
        ex = json.loads(line)
        msgs = ex["messages"]
        usr = next(m for m in msgs if m["role"] == "user")
        ast = next((m for m in msgs if m["role"] == "assistant"), None)
        if ast is None: continue
        ut = parse_user_text(usr["content"])
        m_ch = re.search(r"채널:\s*([^\n]+)", ut)
        m_vn = re.search(r"영상:\s*([^\n]+)", ut)
        if not m_ch or not m_vn: continue
        gt_map[(m_ch.group(1).strip(), m_vn.group(1).strip())] = ast["content"]


# -------------------------
# 2) Pred 로드 + 페어링
# -------------------------
rows = []
for path in tqdm(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True),
                 desc="Pred 로드"):
    try:
        with open(path, encoding="utf-8") as f:
            rec = json.load(f)
    except Exception:
        continue
    if rec.get("orig_channel") != rec.get("cond_channel"):
        continue
    key = (rec["orig_channel"], rec["video_name"])
    if key not in gt_map:
        continue

    gt_text = gt_map[key]
    raw     = rec.get("raw_output", "")
    gt_tok  = count_tokens(gt_text)
    pred_tok = rec.get("completion_tokens") or count_tokens(raw)

    rows.append({
        "channel": key[0], "video": key[1],
        "gt_tok": gt_tok, "pred_tok": pred_tok,
        "ratio":  pred_tok / max(gt_tok, 1),
        "gt_text": gt_text, "raw": raw,
        "instances": rec.get("instances", []),
    })


# -------------------------
# 3) 과생성 심한 샘플 뽑기
# -------------------------
# 짧은 GT + 큰 ratio 조건
cands = [r for r in rows if r["gt_tok"] <= 300 and r["ratio"] >= 3.0]
cands.sort(key=lambda r: r["ratio"], reverse=True)

print(f"\n짧은 GT(≤300) 중 ratio ≥ 3.0 : {len(cands)}개")
print(f"상위 {TOP_K}개 표시\n")

for i, r in enumerate(cands[:TOP_K]):
    print("=" * 80)
    print(f"[{i+1}] channel={r['channel']} | video={r['video']}")
    print(f"    gt_tok={r['gt_tok']}  pred_tok={r['pred_tok']}  ratio={r['ratio']:.1f}")
    print(f"    n_instances={len(r['instances'])}")

    print(f"\n    -- GT ({r['gt_tok']} tok) --")
    print("    " + r["gt_text"][:HEAD_CHARS].replace("\n", "\n    "))

    print(f"\n    -- Pred HEAD ({HEAD_CHARS}자) --")
    print("    " + r["raw"][:HEAD_CHARS].replace("\n", "\n    "))

    print(f"\n    -- Pred TAIL ({TAIL_CHARS}자) --")
    print("    " + r["raw"][-TAIL_CHARS:].replace("\n", "\n    "))

    # 중복 text 카운트
    texts = [inst.get("text", "") for inst in r["instances"] if isinstance(inst, dict)]
    from collections import Counter
    c = Counter(texts)
    top_dup = [(t, n) for t, n in c.most_common(5) if n > 1]
    if top_dup:
        print(f"\n    -- 중복 text top 5 (등장횟수) --")
        for t, n in top_dup:
            print(f"      {n}회: {t[:80]}")
    else:
        print(f"\n    -- 중복 text 없음 (모두 유니크) --")
    print()

The tokenizer you are loading from '/home/taeyoung/nfs-mount/chi2027/model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
GT 로드: 3320it [00:00, 34255.73it/s]
Pred 로드: 100%|██████████| 3038/3038 [00:05<00:00, 527.77it/s]


짧은 GT(≤300) 중 ratio ≥ 3.0 : 563개
상위 10개 표시

[1] channel=ZICO | video=cookin’ in the studio #ZICO #지코_
    gt_tok=4  pred_tok=8489  ratio=2122.2
    n_instances=374

    -- GT (4 tok) --
    {"instances":[]}

    -- Pred HEAD (300자) --
    {"instances":[{"text":"(작곡 중)","start_sec":0.1,"end_sec":2.9},{"text":"Cookin' in the studio","start_sec":0.1,"end_sec":41.3},{"text":"(작곡 중)","start_sec":3.0,"end_sec":3.1},{"text":"(작곡중)","start_sec":3.1,"end_sec":4.1},{"text":"(작곡 중)","start_sec":4.2,"end_sec":4.3},{"text":"(작곡중)","start_sec":4.3

    -- Pred TAIL (600자) --
    0.1},{"text":"(작곡 중)","start_sec":40.1,"end_sec":40.2},{"text":"(작곡중)","start_sec":40.2,"end_sec":40.3},{"text":"(작곡 중)","start_sec":40.3,"end_sec":40.4},{"text":"(작곡중)","start_sec":40.4,"end_sec":40.5},{"text":"(작곡 중)","start_sec":40.5,"end_sec":40.6},{"text":"(작곡중)","start_sec":40.6,"end_sec":40.7},{"text":"(작곡 중)","start_sec":40.7,"end_sec":40.8},{"text":"(작곡중)","start_sec":40.8,"end_sec":40.9},{"text":"(작곡 중)","start_se

In [10]:
# %% 학습 데이터 내 "동일 text 반복" 빈도 조사
import os, json, glob
from collections import Counter
from tqdm.auto import tqdm
import numpy as np

INST_DIR = "./data/6_telop_instances"

paths = glob.glob(os.path.join(INST_DIR, "*", "*.json"))
print(f"영상: {len(paths)}")

per_video = []    # 영상별 (n_inst, max_dup_count, max_dup_text)
all_dup_counts = []  # 전체 영상 걸쳐서 "한 영상 내 동일 text 반복 횟수"

for p in tqdm(paths):
    try:
        with open(p, encoding="utf-8") as f:
            d = json.load(f)
    except Exception:
        continue
    insts = d.get("instances", [])
    if not insts:
        per_video.append((0, 0, ""))
        continue
    
    c = Counter(i["text"] for i in insts if isinstance(i, dict))
    if not c:
        per_video.append((len(insts), 0, ""))
        continue
    top_text, top_n = c.most_common(1)[0]
    per_video.append((len(insts), top_n, top_text))
    
    # 2회 이상 반복된 text들의 반복 횟수 분포
    for t, n in c.items():
        if n >= 2:
            all_dup_counts.append(n)

n_inst    = np.array([x[0] for x in per_video])
max_dup   = np.array([x[1] for x in per_video])

print(f"\n=== 영상당 instance 수 ===")
print(f"  mean={n_inst.mean():.1f} median={np.median(n_inst):.0f} "
      f"p95={np.percentile(n_inst,95):.0f} p99={np.percentile(n_inst,99):.0f} "
      f"max={n_inst.max()}")

print(f"\n=== 영상당 '가장 많이 반복된 텍스트' 등장 횟수 ===")
print(f"  mean={max_dup.mean():.1f} median={np.median(max_dup):.0f} "
      f"p95={np.percentile(max_dup,95):.0f} p99={np.percentile(max_dup,99):.0f} "
      f"max={max_dup.max()}")

print(f"\n=== 동일 text 반복이 심한 영상 비율 ===")
for thr in [5, 10, 30, 50, 100, 200]:
    n = (max_dup >= thr).sum()
    print(f"  max_dup >= {thr:>3}: {n:>6} ({n/len(max_dup)*100:.2f}%)")

# 상위 20개 영상
print(f"\n=== 동일 text 반복 top 20 영상 ===")
idx = np.argsort(-max_dup)[:20]
for i in idx:
    n_inst_v, dup_n, dup_t = per_video[i]
    path = paths[i]
    ch = os.path.basename(os.path.dirname(path))
    vd = os.path.basename(path)[:-5]
    print(f"  [{dup_n:>3}회] {ch[:20]:<20} | {vd[:40]:<40} | '{dup_t[:30]}'")

# 전체 분포
if all_dup_counts:
    arr = np.array(all_dup_counts)
    print(f"\n=== '2회 이상 반복' 발생 건수별 분포 ===")
    print(f"  총 발생: {len(arr)}건")
    for thr in [2, 5, 10, 30, 50, 100]:
        n = (arr >= thr).sum()
        print(f"  반복 >= {thr:>3}: {n:>6}건")

영상: 66392


100%|██████████| 66392/66392 [00:47<00:00, 1401.06it/s]


=== 영상당 instance 수 ===
  mean=57.2 median=33 p95=192 p99=379 max=3824

=== 영상당 '가장 많이 반복된 텍스트' 등장 횟수 ===
  mean=5.7 median=3 p95=21 p99=37 max=610

=== 동일 text 반복이 심한 영상 비율 ===
  max_dup >=   5:  23133 (34.84%)
  max_dup >=  10:  12004 (18.08%)
  max_dup >=  30:   1403 (2.11%)
  max_dup >=  50:    259 (0.39%)
  max_dup >= 100:     35 (0.05%)
  max_dup >= 200:     16 (0.02%)

=== 동일 text 반복 top 20 영상 ===
  [610회] 세바시 강연 Sebasi Talk   | 충격! 제이블랙, ＂재능, 다 필요없다!＂ ｜ 제이블랙 댄서__TfLTV | '새비시'
  [578회] 세바시 강연 Sebasi Talk   | 애정없이 사는 부부？ 부부끼리 잔소리가 많아지는 이유 ｜ 김경일 인지심리 | '새비시'
  [564회] 세바시 강연 Sebasi Talk   | 상처 해방 일지 ： 상처에서 해방되면 생기는 일 #shorts__uyaY | '새비시'
  [520회] 세바시 강연 Sebasi Talk   | 또 다이어트 결심하신 분？ ｜ 김미경 강사, '김미경의 딥마인드' 저자_ | '새비시'
  [505회] 세바시 강연 Sebasi Talk   | 남자 산부인과 의사가 말하는 비애와 비전 ｜ 추성일 헤스티아 여성의원 네 | '새비시'
  [441회] 세바시 강연 Sebasi Talk   | 선한 마음으로 타인 도와주다 뒤통수 맞은 배우 고준, 그럼에도..__9u | '새비시'
  [406회] 세바시 강연 Sebasi Talk   | 우리 쪽팔리게 살지 맙시다 ｜ 김지윤 정치학 박사, 방송인 ｜ 정치 인생 | '새비시'
  [334회] 밍모              

In [11]:
# %% 빈 출력 내는 능력 점검
import os, json, glob, re
from transformers import AutoTokenizer
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test_01"
MODEL_PATH = "./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"


def parse_user_text(c):
    if isinstance(c, str): return c
    if isinstance(c, list):
        return "\n".join(p.get("text", "") for p in c if isinstance(p, dict))
    return ""


tokenizer = AutoTokenizer.from_pretrained(os.path.abspath(MODEL_PATH))

# GT 로드
gt_map = {}
with open(EVAL_JSONL, encoding="utf-8") as f:
    for line in tqdm(f, desc="GT 로드"):
        ex = json.loads(line)
        msgs = ex["messages"]
        usr = next(m for m in msgs if m["role"] == "user")
        ast = next((m for m in msgs if m["role"] == "assistant"), None)
        if ast is None: continue
        ut = parse_user_text(usr["content"])
        m_ch = re.search(r"채널:\s*([^\n]+)", ut)
        m_vn = re.search(r"영상:\s*([^\n]+)", ut)
        if not m_ch or not m_vn: continue
        try:
            gt_obj = json.loads(ast["content"])
            n_gt = len(gt_obj.get("instances", []))
        except Exception:
            n_gt = -1
        gt_map[(m_ch.group(1).strip(), m_vn.group(1).strip())] = n_gt


# Pred 로드 + 매칭
rows = []
for path in tqdm(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True),
                 desc="Pred 로드"):
    try:
        with open(path, encoding="utf-8") as f:
            rec = json.load(f)
    except Exception:
        continue
    if rec.get("orig_channel") != rec.get("cond_channel"):
        continue
    key = (rec["orig_channel"], rec["video_name"])
    if key not in gt_map:
        continue
    rows.append({
        "n_gt":   gt_map[key],
        "n_pred": len(rec.get("instances", [])),
    })

print(f"\n페어: {len(rows)}")


# 집계
n_total      = len(rows)
gt_empty     = [r for r in rows if r["n_gt"] == 0]
pred_empty   = [r for r in rows if r["n_pred"] == 0]
both_empty   = [r for r in rows if r["n_gt"] == 0 and r["n_pred"] == 0]
gt_nonempty  = [r for r in rows if r["n_gt"] > 0]
pred_on_gt_empty = [r["n_pred"] for r in gt_empty]


print(f"\n=== 전체 ===")
print(f"  GT empty (n_gt==0):   {len(gt_empty):>5} ({len(gt_empty)/n_total*100:.2f}%)")
print(f"  Pred empty (n==0):    {len(pred_empty):>5} ({len(pred_empty)/n_total*100:.2f}%)")
print(f"  GT nonempty:          {len(gt_nonempty):>5} ({len(gt_nonempty)/n_total*100:.2f}%)")

print(f"\n=== GT가 empty인 영상에서 모델은? ===")
print(f"  대상: {len(gt_empty)}개")
if gt_empty:
    print(f"  그 중 Pred도 empty:   {len(both_empty):>5} ({len(both_empty)/len(gt_empty)*100:.2f}%)  ← 정답")
    print(f"  Pred도 non-empty:     {len(gt_empty)-len(both_empty):>5} ({(len(gt_empty)-len(both_empty))/len(gt_empty)*100:.2f}%)  ← 과잉생성")

    import numpy as np
    arr = np.array(pred_on_gt_empty)
    print(f"\n  Pred n_instances 분포 (GT empty일 때):")
    print(f"    mean={arr.mean():.1f}  median={np.median(arr):.0f}  "
          f"p50={np.percentile(arr,50):.0f}  p90={np.percentile(arr,90):.0f}  "
          f"p99={np.percentile(arr,99):.0f}  max={arr.max()}")

    for thr in [0, 1, 5, 10, 50, 100, 200]:
        n = (arr >= thr).sum()
        op = ">=" if thr > 0 else "=="
        val = 0 if thr == 0 else thr
        print(f"    n_pred {op} {val:>3}: {n:>5} ({n/len(arr)*100:.2f}%)")

print(f"\n=== GT가 non-empty인데 Pred는 empty? ===")
wrong_empty = [r for r in gt_nonempty if r["n_pred"] == 0]
print(f"  {len(wrong_empty)} / {len(gt_nonempty)} ({len(wrong_empty)/max(len(gt_nonempty),1)*100:.2f}%)")

The tokenizer you are loading from '/home/taeyoung/nfs-mount/chi2027/model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
GT 로드: 3320it [00:00, 20139.43it/s]
Pred 로드: 100%|██████████| 3038/3038 [00:00<00:00, 5258.76it/s]


페어: 3038

=== 전체 ===
  GT empty (n_gt==0):     303 (9.97%)
  Pred empty (n==0):      278 (9.15%)
  GT nonempty:           2735 (90.03%)

=== GT가 empty인 영상에서 모델은? ===
  대상: 303개
  그 중 Pred도 empty:     144 (47.52%)  ← 정답
  Pred도 non-empty:       159 (52.48%)  ← 과잉생성

  Pred n_instances 분포 (GT empty일 때):
    mean=28.2  median=1  p50=1  p90=98  p99=198  max=374
    n_pred ==   0:   303 (100.00%)
    n_pred >=   1:   159 (52.48%)
    n_pred >=   5:   101 (33.33%)
    n_pred >=  10:    95 (31.35%)
    n_pred >=  50:    72 (23.76%)
    n_pred >= 100:    30 (9.90%)
    n_pred >= 200:     3 (0.99%)

=== GT가 non-empty인데 Pred는 empty? ===
  134 / 2735 (4.90%)


In [12]:
# %% GT empty일 때 Pred 도배 vs Pred empty 입력 차이 분석
import os, json, glob, re
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test_01"
INST_DIR   = "./data/6_telop_instances"


def parse_user_text(c):
    if isinstance(c, str): return c
    if isinstance(c, list):
        return "\n".join(p.get("text", "") for p in c if isinstance(p, dict))
    return ""


# eval.jsonl 파싱: user_text 통째로 보관
eval_map = {}
with open(EVAL_JSONL, encoding="utf-8") as f:
    for line in tqdm(f, desc="eval 로드"):
        ex = json.loads(line)
        msgs = ex["messages"]
        usr = next(m for m in msgs if m["role"] == "user")
        ast = next((m for m in msgs if m["role"] == "assistant"), None)
        if ast is None: continue
        ut = parse_user_text(usr["content"])
        m_ch = re.search(r"채널:\s*([^\n]+)", ut)
        m_vn = re.search(r"영상:\s*([^\n]+)", ut)
        if not m_ch or not m_vn: continue
        try:
            gt_obj = json.loads(ast["content"])
            n_gt = len(gt_obj.get("instances", []))
        except Exception:
            continue
        # STT 세그먼트 수 / 총 발화길이 / 영상 길이
        stt_lines = [l for l in ut.splitlines() if re.match(r"\[\d", l)]
        stt_chars = sum(len(l) for l in stt_lines)
        max_end = 0.0
        for l in stt_lines:
            mt = re.match(r"\[(\d+\.?\d*)-(\d+\.?\d*)\]", l)
            if mt:
                max_end = max(max_end, float(mt.group(2)))
        eval_map[(m_ch.group(1).strip(), m_vn.group(1).strip())] = {
            "n_gt": n_gt, "n_stt": len(stt_lines), "stt_chars": stt_chars,
            "vid_len": max_end,
        }


# 채널별 평균 텔롭 밀도 (stats_df와 동일 계산)
def channel_avg_density(ch):
    ch_dir = os.path.join(INST_DIR, ch)
    if not os.path.isdir(ch_dir): return None
    paths = glob.glob(os.path.join(glob.escape(ch_dir), "*.json"))
    if not paths: return None
    d_sum, n_vid = 0.0, 0
    for p in paths:
        try:
            with open(p, encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue
        insts = data.get("instances", [])
        n_vid += 1
        if not insts: continue
        ends = np.array([i["end_sec"] for i in insts])
        starts = np.array([i["start_sec"] for i in insts])
        vl = float(ends.max()) if len(ends) else 0
        if vl > 0:
            d_sum += float(np.clip(ends - starts, 0, None).sum()) / vl
    return d_sum / n_vid if n_vid else None


# GT empty인 영상만 Pred 수집 + 그룹핑
pred_map = {}
for path in tqdm(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True),
                 desc="Pred 로드"):
    try:
        with open(path, encoding="utf-8") as f: rec = json.load(f)
    except Exception: continue
    if rec.get("orig_channel") != rec.get("cond_channel"): continue
    key = (rec["orig_channel"], rec["video_name"])
    if key not in eval_map or eval_map[key]["n_gt"] != 0: continue
    pred_map[key] = len(rec.get("instances", []))


# 채널 밀도 캐시
ch_set = {k[0] for k in pred_map}
print(f"\n채널 밀도 계산 ({len(ch_set)}개)...")
ch_dens = {ch: channel_avg_density(ch) for ch in tqdm(ch_set)}


# 그룹 나누기: Pred empty / mild (1-10) / heavy (>=50)
groups = {"pred=0": [], "pred 1~10": [], "pred 11~49": [], "pred>=50": []}
for k, n_pred in pred_map.items():
    info = eval_map[k]
    dens = ch_dens.get(k[0])
    item = {"key": k, "n_pred": n_pred, "n_stt": info["n_stt"],
            "stt_chars": info["stt_chars"], "vid_len": info["vid_len"],
            "ch_density": dens}
    if n_pred == 0: groups["pred=0"].append(item)
    elif n_pred <= 10: groups["pred 1~10"].append(item)
    elif n_pred < 50: groups["pred 11~49"].append(item)
    else: groups["pred>=50"].append(item)


# 그룹별 통계
print(f"\n=== GT empty (n={sum(len(v) for v in groups.values())}개)에서 Pred 분포별 입력 특성 ===")
print(f"  {'group':<12} {'n':>5} "
      f"{'n_stt_mean':>11} {'stt_chars_mean':>15} "
      f"{'vid_len_mean':>13} {'ch_dens_mean':>13}")
for name, items in groups.items():
    if not items:
        print(f"  {name:<12} {0:>5}")
        continue
    n_stt   = np.mean([i["n_stt"] for i in items])
    stt_ch  = np.mean([i["stt_chars"] for i in items])
    vl      = np.mean([i["vid_len"] for i in items])
    dens    = [i["ch_density"] for i in items if i["ch_density"] is not None]
    dens_m  = np.mean(dens) if dens else float("nan")
    print(f"  {name:<12} {len(items):>5} "
          f"{n_stt:>11.1f} {stt_ch:>15.1f} "
          f"{vl:>13.1f} {dens_m:>13.4f}")


# 채널별로 GT empty에서 과잉생성 경향
print(f"\n=== GT empty일 때 채널별 Pred 분포 (GT empty 영상 3개 이상인 채널만, 과잉률 내림차순) ===")
by_ch = defaultdict(list)
for k, n_pred in pred_map.items():
    by_ch[k[0]].append(n_pred)

ch_stats = []
for ch, preds in by_ch.items():
    if len(preds) < 3: continue
    over = sum(1 for p in preds if p > 0) / len(preds)
    ch_stats.append((ch, len(preds), over, np.mean(preds), np.max(preds), ch_dens.get(ch)))
ch_stats.sort(key=lambda x: -x[2])

print(f"  {'channel':<35} {'n':>4} {'over_rate':>10} {'pred_mean':>10} {'pred_max':>9} {'ch_dens':>8}")
for ch, n, over, pm, pmax, dens in ch_stats[:25]:
    dens_s = f"{dens:.3f}" if dens is not None else "N/A"
    print(f"  {ch[:33]:<35} {n:>4} {over*100:>9.1f}% {pm:>10.1f} {pmax:>9.0f} {dens_s:>8}")

print(f"\n  === 하위 10개 (GT empty에서 Pred도 empty 잘 내는 채널) ===")
for ch, n, over, pm, pmax, dens in ch_stats[-10:]:
    dens_s = f"{dens:.3f}" if dens is not None else "N/A"
    print(f"  {ch[:33]:<35} {n:>4} {over*100:>9.1f}% {pm:>10.1f} {pmax:>9.0f} {dens_s:>8}")

eval 로드: 3320it [00:00, 13789.28it/s]
Pred 로드: 100%|██████████| 3038/3038 [00:00<00:00, 5281.14it/s]



채널 밀도 계산 (161개)...


100%|██████████| 161/161 [00:04<00:00, 32.60it/s]


=== GT empty (n=303개)에서 Pred 분포별 입력 특성 ===
  group            n  n_stt_mean  stt_chars_mean  vid_len_mean  ch_dens_mean
  pred=0         144         4.6           115.2          17.6        0.3179
  pred 1~10       65         4.8           108.5          24.3        0.9727
  pred 11~49      22         6.0           191.6          22.0        1.0101
  pred>=50        72         6.0           147.9          32.8        0.7526

=== GT empty일 때 채널별 Pred 분포 (GT empty 영상 3개 이상인 채널만, 과잉률 내림차순) ===
  channel                                n  over_rate  pred_mean  pred_max  ch_dens
  DOM Studio                             3     100.0%       72.3       140    0.165
  1MILLION Dance Studio                  3     100.0%        1.0         1    1.416
  ETTV 이티티비                              3     100.0%       43.7       129    0.011
  imlisarhee                             3     100.0%       43.0       124    0.644
  DJ SODA OFFICIAL                       3     100.0%       50.7       116    0.122

In [13]:
# %% 인스턴스 duration 분포 (0.1초 단위)
import os, json
import numpy as np

INST_DIR = "./data/6_telop_instances"

durations = []

for ch in sorted(os.listdir(INST_DIR)):
    ch_dir = os.path.join(INST_DIR, ch)
    if not os.path.isdir(ch_dir):
        continue
    for fname in os.listdir(ch_dir):
        if not fname.endswith(".json"):
            continue
        with open(os.path.join(ch_dir, fname), "r", encoding="utf-8") as f:
            data = json.load(f)
        for inst in data.get("instances", []):
            dur = inst["end_sec"] - inst["start_sec"]
            durations.append(dur)

durations = np.array(durations)
total = len(durations)

print(f"전체 인스턴스: {total:,}")
print(f"평균: {durations.mean():.2f}s, 중앙값: {np.median(durations):.2f}s")
print(f"최소: {durations.min():.2f}s, 최대: {durations.max():.2f}s")

print(f"\n{'구간':>12}  {'개수':>10}  {'비율':>8}  {'누적':>8}")
print(f"{'─'*12}  {'─'*10}  {'─'*8}  {'─'*8}")
cumul = 0
for lo in np.arange(0, 2.0, 0.1):
    hi = lo + 0.1
    cnt = int(((durations >= lo) & (durations < hi)).sum())
    cumul += cnt
    pct = cnt / total * 100
    cpct = cumul / total * 100
    print(f"  {lo:.1f}~{hi:.1f}초   {cnt:>10,}    {pct:>5.1f}%   {cpct:>5.1f}%")

over_2 = int((durations >= 2.0).sum())
cumul += over_2
print(f"  ≥2.0초       {over_2:>10,}    {over_2/total*100:>5.1f}%   {cumul/total*100:>5.1f}%")

전체 인스턴스: 3,797,013
평균: 2.37s, 중앙값: 0.30s
최소: 0.10s, 최대: 180.00s

          구간          개수        비율        누적
────────────  ──────────  ────────  ────────
  0.0~0.1초      711,462     18.7%    18.7%
  0.1~0.2초    1,016,725     26.8%    45.5%
  0.2~0.3초      154,869      4.1%    49.6%
  0.3~0.4초      133,563      3.5%    53.1%
  0.4~0.5초       45,450      1.2%    54.3%
  0.5~0.6초      128,972      3.4%    57.7%
  0.6~0.7초       93,046      2.5%    60.2%
  0.7~0.8초       75,650      2.0%    62.1%
  0.8~0.9초       89,495      2.4%    64.5%
  0.9~1.0초       36,791      1.0%    65.5%
  1.0~1.1초      109,010      2.9%    68.3%
  1.1~1.2초       78,270      2.1%    70.4%
  1.2~1.3초       62,276      1.6%    72.0%
  1.3~1.4초       70,282      1.9%    73.9%
  1.4~1.5초       79,413      2.1%    76.0%
  1.5~1.6초       78,229      2.1%    78.0%
  1.6~1.7초       56,782      1.5%    79.5%
  1.7~1.8초       44,378      1.2%    80.7%
  1.8~1.9초       48,966      1.3%    82.0%
  1.9~2.0초       19,057     

In [14]:
# %% 짧은 인스턴스 (≤0.2초) 실제 내용 확인
import os, json, random
from collections import Counter

INST_DIR = "./data/6_telop_instances"

short_instances = []  # duration ≤ 0.2초

for ch in sorted(os.listdir(INST_DIR)):
    ch_dir = os.path.join(INST_DIR, ch)
    if not os.path.isdir(ch_dir):
        continue
    for fname in os.listdir(ch_dir):
        if not fname.endswith(".json"):
            continue
        with open(os.path.join(ch_dir, fname), "r", encoding="utf-8") as f:
            data = json.load(f)
        for inst in data.get("instances", []):
            dur = inst["end_sec"] - inst["start_sec"]
            if dur <= 0.2:
                short_instances.append({
                    "channel": ch,
                    "video": fname[:-5][:50],
                    "text": inst["text"],
                    "start": inst["start_sec"],
                    "end": inst["end_sec"],
                    "dur": round(dur, 2),
                })

print(f"≤0.2초 인스턴스: {len(short_instances):,}개\n")

# 텍스트 길이 분포
text_lens = [len(s["text"]) for s in short_instances]
print(f"텍스트 길이 — 평균: {sum(text_lens)/len(text_lens):.1f}, 최대: {max(text_lens)}")

# 텍스트 내용 빈도
texts = [s["text"] for s in short_instances]
counter = Counter(texts)
print(f"고유 텍스트 수: {len(counter):,}\n")

print(f"📋 가장 많이 등장하는 텍스트 (상위 30개)")
for text, cnt in counter.most_common(30):
    print(f"  {cnt:>8,}회  {text!r}")

# 랜덤 50개 예시
print(f"\n📋 랜덤 50개 예시")
random.seed(42)
samples = random.sample(short_instances, min(50, len(short_instances)))
for s in samples:
    print(f"  [{s['start']:.1f}-{s['end']:.1f}] ({s['dur']}s) {s['channel']}/{s['video']}")
    print(f"    {s['text']!r}")

≤0.2초 인스턴스: 1,730,989개

텍스트 길이 — 평균: 5.5, 최대: 169
고유 텍스트 수: 852,444

📋 가장 많이 등장하는 텍스트 (상위 30개)
    10,354회  '2'
    10,193회  '0'
     9,964회  '-'
     7,259회  '1'
     6,320회  '3'
     6,283회  '→'
     5,844회  '7'
     5,697회  '*'
     4,699회  '.'
     4,607회  'T'
     4,453회  '4'
     4,350회  '•'
     4,029회  '5'
     3,917회  'X'
     3,908회  '※'
     3,765회  '이'
     3,678회  '+'
     3,545회  '–'
     3,439회  'E'
     3,426회  'A'
     2,958회  'V'
     2,904회  'O'
     2,766회  '—'
     2,722회  'D'
     2,579회  '6'
     2,436회  '9'
     2,418회  '×'
     2,391회  'S'
     2,265회  '8'
     2,217회  'N'

📋 랜덤 50개 예시
  [45.0-45.1] (0.1s) 오빠두엑셀 l 엑셀 강의 대표채널/와.. 이거 엑셀 맞나요？!😲 ＂사진→시트＂ 변환, 이제 10초면 됩니다! #shorts_
    '※'
  [45.8-45.9] (0.1s) KBS Entertain_ 깔깔티비/#shorts 내가 사랑한 승기야~ ｜ KBS 080322 방송__fWfg7ksDoc8
    '서유 특진'
  [31.5-31.6] (0.1s) BANGTANTV/'i wonder... (with Jung Kook of BTS)' Dance Video🕺
    'THESTREE'
  [14.4-14.5] (0.1s) 찰스엔터/뚜껑 좀 따줄래..？__5GxH3k0gWrY
    '이러다 똥 나오겠어요.'
  [16.1-16.2] 

In [1]:
# %% Gradio: 짧은 인스턴스 실제 프레임 확인
import os, json, random, glob
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import gradio as gr
import subprocess
subprocess.run("fuser -k 7860/tcp", shell=True, stderr=subprocess.DEVNULL)

INST_DIR = "./data/6_telop_instances"
OCR_DIR = "./data/3_ocr_results"
FRAME_DIR = "./data/2_frame_files"
FPS = 10


def _loads(x):
    return json.loads(x) if isinstance(x, str) else x


def collect_channel(ch):
    ch_dir = os.path.join(INST_DIR, ch)
    results = []
    for fname in os.listdir(ch_dir):
        if not fname.endswith(".json"):
            continue
        with open(os.path.join(ch_dir, fname), "r", encoding="utf-8") as f:
            data = json.load(f)
        vname = fname[:-5]
        for inst in data.get("instances", []):
            dur = inst["end_sec"] - inst["start_sec"]
            if dur <= 0.3:
                results.append({
                    "channel": ch,
                    "video": vname,
                    "text": inst["text"],
                    "start_sec": inst["start_sec"],
                    "end_sec": inst["end_sec"],
                    "dur": round(dur, 2),
                })
    return results


# 병렬 수집
print("📦 인스턴스 수집 중...")
channels = [ch for ch in sorted(os.listdir(INST_DIR)) if os.path.isdir(os.path.join(INST_DIR, ch))]

all_short = []
with ProcessPoolExecutor(max_workers=32) as pool:
    futures = {pool.submit(collect_channel, ch): ch for ch in channels}
    for f in tqdm(as_completed(futures), total=len(futures), desc="수집"):
        all_short.extend(f.result())

# duration별 그룹핑
by_dur = defaultdict(list)
for s in all_short:
    by_dur[s["dur"]].append(s)

for k, v in sorted(by_dur.items()):
    print(f"  {k}초: {len(v):,}개")
print(f"✅ 전체 ≤0.3초 인스턴스: {len(all_short):,}개")


def draw_bbox_on_frame(channel, video, frame_num, target_text):
    frame_path = os.path.join(FRAME_DIR, channel, video, f"frame_{frame_num:06d}.jpg")
    if not os.path.exists(frame_path):
        pattern = os.path.join(FRAME_DIR, channel, video, f"*{frame_num:06d}*")
        matches = glob.glob(pattern)
        if matches:
            frame_path = matches[0]
        else:
            return None, f"프레임 이미지 없음: frame_{frame_num}"

    img = Image.open(frame_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    ocr_pq = os.path.join(OCR_DIR, channel, video + ".parquet")
    if not os.path.exists(ocr_pq):
        return img, "OCR parquet 없음"

    df = pd.read_parquet(ocr_pq)
    row = df[df["frame_num"] == frame_num]
    if len(row) == 0:
        return img, "해당 프레임에 OCR 데이터 없음"

    row = row.iloc[0]
    texts = _loads(row["ocr_texts"])
    xywhas = _loads(row["ocr_xywha"])

    import math
    for text, xywha in zip(texts, xywhas):
        cx, cy, w, h, a = xywha
        rad = math.radians(a)
        cos_a, sin_a = math.cos(rad), math.sin(rad)
        dx = [(-w/2, -h/2), (w/2, -h/2), (w/2, h/2), (-w/2, h/2)]
        pts = [(int(cx + d[0]*cos_a - d[1]*sin_a), int(cy + d[0]*sin_a + d[1]*cos_a)) for d in dx]

        if text == target_text:
            color, width = "red", 3
        else:
            color, width = "green", 1

        for j in range(len(pts)):
            draw.line([pts[j], pts[(j+1) % len(pts)]], fill=color, width=width)
        try:
            draw.text((pts[0][0], pts[0][1] - 15), text[:30], fill=color)
        except:
            pass

    return img, f"frame={frame_num}, texts={len(texts)}개"

def show_random(dur_choice):
    dur_val = round(float(dur_choice.replace("초", "")), 1)
    pool = by_dur.get(dur_val, [])
    if not pool:
        return None, None, None, f"duration={dur_val}초: 0개"

    s = random.choice(pool)
    start_frame = int(s["start_sec"] * FPS)
    prev_frame = max(0, start_frame - 1)
    next_frame = start_frame + 1

    img_prev, info_prev = draw_bbox_on_frame(s["channel"], s["video"], prev_frame, s["text"])
    img_curr, info_curr = draw_bbox_on_frame(s["channel"], s["video"], start_frame, s["text"])
    img_next, info_next = draw_bbox_on_frame(s["channel"], s["video"], next_frame, s["text"])

    info = (
        f"채널: {s['channel']}\n"
        f"영상: {s['video'][:80]}\n"
        f"텍스트: {s['text']!r}\n"
        f"시간: [{s['start_sec']:.1f}-{s['end_sec']:.1f}] ({s['dur']}s)\n"
        f"pool: {len(pool):,}개 중 랜덤\n"
        f"\n좌: 이전프레임 ({info_prev})\n중: 해당프레임 ({info_curr})\n우: 다음프레임 ({info_next})"
    )

    return img_prev, img_curr, img_next, info


with gr.Blocks(title="짧은 인스턴스 확인") as demo:
    gr.Markdown("## 짧은 인스턴스 실제 프레임 확인 (bbox 포함)")

    with gr.Row():
        dur_dropdown = gr.Dropdown(
            choices=["0.1초", "0.2초", "0.3초"],
            value="0.1초",
            label="Duration 필터"
        )
        rand_btn = gr.Button("🎲 랜덤 샘플 보기", variant="primary")

    with gr.Row():
        img_prev = gr.Image(label="이전 프레임", height=400)
        img_curr = gr.Image(label="해당 프레임 (빨강=타겟)", height=400)
        img_next = gr.Image(label="다음 프레임", height=400)

    info_box = gr.Textbox(label="정보", lines=8, interactive=False)

    rand_btn.click(show_random, inputs=[dur_dropdown], outputs=[img_prev, img_curr, img_next, info_box])

demo.launch(server_name="0.0.0.0", server_port=7860)

/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 인스턴스 수집 중...


수집: 100%|██████████| 664/664 [00:02<00:00, 229.87it/s]


  0.1초: 1,616,366개
  0.2초: 202,469개
  0.3초: 64,221개
✅ 전체 ≤0.3초 인스턴스: 1,883,056개
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
import os, json, glob, re

OUTPUT_DIR = "./data/7_qwen_test_01"
EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"

# 1) pred 수집
pred_map = {}  # (ch, video) → n_instances
for path in glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    if rec.get("orig_channel") != rec.get("cond_channel"):
        continue
    key = (rec["orig_channel"], rec["video_name"])
    pred_map[key] = len(rec.get("instances", []))

# 2) GT 수집 (pred와 매칭되는 것만)
gt_map = {}
with open(EVAL_JSONL, "r") as f:
    for line in f:
        ex = json.loads(line)
        ch = ex["metadata"]["channel"]
        vn = ex["metadata"]["video"].rsplit("__", 1)[0]
        key = (ch, vn)
        if key in pred_map:
            gt_map[key] = ex["metadata"]["num_instances"]

# 3) 비교
gt_empty = sum(1 for v in gt_map.values() if v == 0)
pred_empty = sum(1 for k in gt_map if pred_map[k] == 0)

# 교차 분석
both_empty = sum(1 for k in gt_map if gt_map[k] == 0 and pred_map[k] == 0)
gt_empty_pred_not = sum(1 for k in gt_map if gt_map[k] == 0 and pred_map[k] > 0)
gt_not_pred_empty = sum(1 for k in gt_map if gt_map[k] > 0 and pred_map[k] == 0)
both_not_empty = sum(1 for k in gt_map if gt_map[k] > 0 and pred_map[k] > 0)

total = len(gt_map)
print(f"매칭된 샘플: {total:,}개\n")
print(f"{'':>20} {'Pred 빈':>12} {'Pred 있음':>12} {'합계':>12}")
print(f"{'GT 빈':>20} {both_empty:>12,} {gt_empty_pred_not:>12,} {gt_empty:>12,}")
print(f"{'GT 있음':>20} {gt_not_pred_empty:>12,} {both_not_empty:>12,} {total-gt_empty:>12,}")
print(f"{'합계':>20} {pred_empty:>12,} {total-pred_empty:>12,} {total:>12,}")

print(f"\nGT 빈 → Pred도 빈: {both_empty}/{gt_empty} ({both_empty/max(gt_empty,1)*100:.1f}%)")
print(f"GT 빈 → Pred 생성: {gt_empty_pred_not}/{gt_empty} ({gt_empty_pred_not/max(gt_empty,1)*100:.1f}%) ← 환각")
print(f"GT 있음 → Pred 빈: {gt_not_pred_empty}/{total-gt_empty} ({gt_not_pred_empty/max(total-gt_empty,1)*100:.1f}%) ← 누락")

매칭된 샘플: 3,038개

                           Pred 빈      Pred 있음           합계
                GT 빈          144          159          303
               GT 있음          134        2,601        2,735
                  합계          278        2,760        3,038

GT 빈 → Pred도 빈: 144/303 (47.5%)
GT 빈 → Pred 생성: 159/303 (52.5%) ← 환각
GT 있음 → Pred 빈: 134/2735 (4.9%) ← 누락


In [4]:
import os, json, glob

OUTPUT_DIR = "./data/7_qwen_test_01"
EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
TRAIN_JSONL = "./data/7_qwen_dataset_train.jsonl"

# 1) pred 수집
pred_map = {}
for path in glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    if rec.get("orig_channel") != rec.get("cond_channel"):
        continue
    key = (rec["orig_channel"], rec["video_name"])
    pred_map[key] = len(rec.get("instances", []))

# 2) eval GT 수집
eval_gt = {}
with open(EVAL_JSONL, "r") as f:
    for line in f:
        ex = json.loads(line)
        ch = ex["metadata"]["channel"]
        vn = ex["metadata"]["video"].rsplit("__", 1)[0]
        key = (ch, vn)
        if key in pred_map:
            eval_gt[key] = ex["metadata"]["num_instances"]

# 3) 환각 채널 목록
halluc_channels = set()
for k in eval_gt:
    if eval_gt[k] == 0 and pred_map[k] > 0:
        halluc_channels.add(k[0])

# 4) 해당 채널의 train+eval 전체 GT에서 빈/비지않은 비율
from collections import defaultdict
channel_stats = defaultdict(lambda: {"empty": 0, "nonempty": 0})

for jsonl_path in [TRAIN_JSONL, EVAL_JSONL]:
    with open(jsonl_path, "r") as f:
        for line in f:
            ex = json.loads(line)
            ch = ex["metadata"]["channel"]
            if ch not in halluc_channels:
                continue
            if ex["metadata"]["num_instances"] == 0:
                channel_stats[ch]["empty"] += 1
            else:
                channel_stats[ch]["nonempty"] += 1

# 5) 출력
print(f"환각 생성한 채널: {len(halluc_channels)}개\n")
print(f"{'채널':<35} {'빈GT':>6} {'있음GT':>6} {'전체':>6} {'빈비율':>8}")
print(f"{'─'*35} {'─'*6} {'─'*6} {'─'*6} {'─'*8}")

rows = []
for ch in sorted(halluc_channels):
    s = channel_stats[ch]
    total = s["empty"] + s["nonempty"]
    pct = s["empty"] / total * 100 if total > 0 else 0
    rows.append((ch, s["empty"], s["nonempty"], total, pct))

rows.sort(key=lambda x: x[4], reverse=True)
for ch, emp, nemp, tot, pct in rows:
    print(f"  {ch:<35} {emp:>5} {nemp:>5} {tot:>5} {pct:>7.1f}%")

# 요약
total_emp = sum(r[1] for r in rows)
total_nemp = sum(r[2] for r in rows)
total_all = total_emp + total_nemp
print(f"\n{'합계':<35} {total_emp:>5} {total_nemp:>5} {total_all:>5} {total_emp/total_all*100:>7.1f}%")

환각 생성한 채널: 115개

채널                                     빈GT   있음GT     전체      빈비율
─────────────────────────────────── ────── ────── ────── ────────
  MOVE Dance Studio                      64    36   100    64.0%
  NCT WISH                               62    38   100    62.0%
  딩가딩가 스튜디오 DGDG Studio                  59    41   100    59.0%
  Puuung 퍼엉                              57    43   100    57.0%
  Dreamcatcher official                  56    44   100    56.0%
  TWICE                                  56    44   100    56.0%
  Official ARTMS                         55    45   100    55.0%
  IVE                                    52    48   100    52.0%
  슈앤트리 SHU AND TREE                      51    49   100    51.0%
  KISS OF LIFE                           49    51   100    49.0%
  LE SSERAFIM                            49    51   100    49.0%
  DaftTaengk                             48    52   100    48.0%
  NCT                                    48    52   100    48.0%
  ARTB

In [ ]:
흠.. 지금은 훈련에 1)채널명, 2)영상명, 3)STT, 4)GT telop을 넣고, inference시 pred Telop을 생성하도록 하는거잖아? 그러면 뭔가 훈련데이터 구성에 뭔가가 빠져서 채널 내에서도 다른 경우

In [5]:
import os, json, glob, re
import numpy as np

OUTPUT_DIR = "./data/7_qwen_test_01"
EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"

# pred 수집
pred_map = {}
for path in glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    if rec.get("orig_channel") != rec.get("cond_channel"):
        continue
    key = (rec["orig_channel"], rec["video_name"])
    pred_map[key] = {
        "n_instances": len(rec.get("instances", [])),
        "completion_tokens": rec.get("completion_tokens", 0),
    }

# eval GT 수집
correct_empty = []  # GT빈 + Pred빈
halluc_empty = []   # GT빈 + Pred생성

with open(EVAL_JSONL, "r") as f:
    for line in f:
        ex = json.loads(line)
        ch = ex["metadata"]["channel"]
        vn = ex["metadata"]["video"].rsplit("__", 1)[0]
        key = (ch, vn)
        if key not in pred_map:
            continue
        if ex["metadata"]["num_instances"] != 0:
            continue

        # STT 세그먼트 수, user_text 길이
        user_text = ex["messages"][1]["content"]
        stt_lines = [l for l in user_text.split("\n") if l.startswith("[")]
        
        entry = {
            "channel": ch,
            "video": vn,
            "n_stt": len(stt_lines),
            "user_text_len": len(user_text),
            "pred_instances": pred_map[key]["n_instances"],
            "pred_tokens": pred_map[key]["completion_tokens"],
        }

        if pred_map[key]["n_instances"] == 0:
            correct_empty.append(entry)
        else:
            halluc_empty.append(entry)

print(f"올바르게 빈: {len(correct_empty)}개")
print(f"환각 생성: {len(halluc_empty)}개")

print(f"\n{'':>20} {'올바르게 빈':>15} {'환각':>15}")
print(f"{'─'*20} {'─'*15} {'─'*15}")

ce_stt = [e["n_stt"] for e in correct_empty]
he_stt = [e["n_stt"] for e in halluc_empty]
print(f"{'STT 세그먼트 수':>20} {'평균 '+str(round(np.mean(ce_stt),1)):>15} {'평균 '+str(round(np.mean(he_stt),1)):>15}")
print(f"{'':>20} {'중앙 '+str(round(np.median(ce_stt),1)):>15} {'중앙 '+str(round(np.median(he_stt),1)):>15}")

ce_stt0 = sum(1 for e in correct_empty if e["n_stt"] == 0)
he_stt0 = sum(1 for e in halluc_empty if e["n_stt"] == 0)
print(f"{'STT 없음 (0개)':>20} {ce_stt0:>10} ({ce_stt0/len(correct_empty)*100:.1f}%) {he_stt0:>10} ({he_stt0/len(halluc_empty)*100:.1f}%)")

ce_len = [e["user_text_len"] for e in correct_empty]
he_len = [e["user_text_len"] for e in halluc_empty]
print(f"{'user_text 길이':>20} {'평균 '+str(round(np.mean(ce_len),0)):>15} {'평균 '+str(round(np.mean(he_len),0)):>15}")

ce_ptok = [e["pred_tokens"] for e in halluc_empty]
print(f"\n환각 생성물 토큰 — 평균: {np.mean(ce_ptok):.0f}, 중앙: {np.median(ce_ptok):.0f}, 최대: {np.max(ce_ptok)}")

# 환각 샘플 5개 예시
print(f"\n📋 환각 샘플 (상위 5개, pred_tokens 높은 순)")
halluc_empty.sort(key=lambda x: x["pred_tokens"], reverse=True)
for e in halluc_empty[:5]:
    print(f"  {e['channel']} / {e['video'][:50]}")
    print(f"    STT={e['n_stt']}개, pred_instances={e['pred_instances']}, pred_tokens={e['pred_tokens']}")

올바르게 빈: 144개
환각 생성: 159개

                              올바르게 빈              환각
──────────────────── ─────────────── ───────────────
          STT 세그먼트 수          평균 4.6          평균 5.5
                              중앙 3.5          중앙 4.0
         STT 없음 (0개)          0 (0.0%)          0 (0.0%)
        user_text 길이        평균 195.0        평균 214.0

환각 생성물 토큰 — 평균: 1096, 중앙: 637, 최대: 8489

📋 환각 샘플 (상위 5개, pred_tokens 높은 순)
  ZICO / cookin’ in the studio #ZICO #지코_
    STT=14개, pred_instances=374, pred_tokens=8489
  ILLIT / N★T CUTE #LE_SSERAFIM #HUHYUNJIN ANYMORE 🍝😜#NOT_CU
    STT=1개, pred_instances=264, pred_tokens=5745
  허팝Heopop / 100만번만 흔들면 초코볼이 됩니다 ㅋㅋㅋ
    STT=23개, pred_instances=198, pred_tokens=4291
  STAYC / #엔믹스 에게 마법에 걸린 것처럼 빠져든 밤!🌙 #해원 님과 함께한 #Bubble_Chal
    STT=5개, pred_instances=200, pred_tokens=3939
 고기남자 / 포르게타 괴물…
    STT=4개, pred_instances=178, pred_tokens=3859


In [6]:
import os, json, glob
import numpy as np

OUTPUT_DIR = "./data/7_qwen_test_01"

gt_durs = []
pred_durs = []

for path in glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    if rec.get("orig_channel") != rec.get("cond_channel"):
        continue
    for inst in rec.get("instances", []):
        dur = inst.get("end_sec", 0) - inst.get("start_sec", 0)
        pred_durs.append(dur)

pred_durs = np.array(pred_durs)
print(f"Pred 인스턴스: {len(pred_durs):,}개")
print(f"평균: {pred_durs.mean():.2f}s, 중앙값: {np.median(pred_durs):.2f}s")

print(f"\n{'구간':>12}  {'개수':>10}  {'비율':>8}  {'누적':>8}")
print(f"{'─'*12}  {'─'*10}  {'─'*8}  {'─'*8}")
cumul = 0
for lo in np.arange(0, 2.0, 0.1):
    hi = lo + 0.1
    cnt = int(((pred_durs >= lo) & (pred_durs < hi)).sum())
    cumul += cnt
    pct = cnt / len(pred_durs) * 100
    cpct = cumul / len(pred_durs) * 100
    print(f"  {lo:.1f}~{hi:.1f}초   {cnt:>10,}    {pct:>5.1f}%   {cpct:>5.1f}%")

over_2 = int((pred_durs >= 2.0).sum())
cumul += over_2
print(f"  ≥2.0초       {over_2:>10,}    {over_2/len(pred_durs)*100:>5.1f}%   {cumul/len(pred_durs)*100:>5.1f}%")

Pred 인스턴스: 257,220개
평균: 1.29s, 중앙값: 0.10s

          구간          개수        비율        누적
────────────  ──────────  ────────  ────────
  0.0~0.1초       98,207     38.2%    38.2%
  0.1~0.2초      109,818     42.7%    80.9%
  0.2~0.3초        1,121      0.4%    81.3%
  0.3~0.4초        1,011      0.4%    81.7%
  0.4~0.5초          449      0.2%    81.9%
  0.5~0.6초        2,113      0.8%    82.7%
  0.6~0.7초        2,248      0.9%    83.6%
  0.7~0.8초        2,237      0.9%    84.4%
  0.8~0.9초        2,646      1.0%    85.5%
  0.9~1.0초        1,321      0.5%    86.0%
  1.0~1.1초        5,089      2.0%    88.0%
  1.1~1.2초        2,634      1.0%    89.0%
  1.2~1.3초        2,085      0.8%    89.8%
  1.3~1.4초        2,490      1.0%    90.8%
  1.4~1.5초        2,579      1.0%    91.8%
  1.5~1.6초        2,431      0.9%    92.7%
  1.6~1.7초        1,695      0.7%    93.4%
  1.7~1.8초        1,359      0.5%    93.9%
  1.8~1.9초        1,321      0.5%    94.4%
  1.9~2.0초          518      0.2%    94.6%
  ≥2.0초

In [14]:
# %% 토큰 길이 분포 비교: GT vs 생성된 JSON (페어 기반)
import os
import json
import glob
import re
import numpy as np
from transformers import AutoTokenizer
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test"
MODEL_PATH = "./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"

print("🔧 tokenizer 로딩...")
MODEL_PATH_ABS = os.path.abspath(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH_ABS)

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

def parse_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(part.get("text", "") for part in msg_content if isinstance(part, dict))
    return ""

# -------------------------
# 2) 완료된 JSON 수집
# -------------------------
print("\n📦 완료된 생성 JSON 수집...")
json_files = glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True)

done_map = {}
for path in tqdm(json_files, desc="생성 JSON 파싱"):
    try:
        with open(path, "r", encoding="utf-8") as f:
            rec = json.load(f)
    except Exception:
        continue

    orig_ch = rec.get("orig_channel")
    cond_ch = rec.get("cond_channel")
    video   = rec.get("video_name")

    if orig_ch != cond_ch:
        continue

    pred_tok = rec.get("completion_tokens")
    if pred_tok is None:
        raw = rec.get("raw_output", "")
        if not raw:
            continue
        pred_tok = count_tokens(raw)

    done_map[(orig_ch, video)] = {
        "pred_tok": pred_tok,
        "pred_inst": len(rec.get("instances", [])),
    }

print(f"✅ 완료된 생성물: {len(done_map)}")

# -------------------------
# 3) GT 매칭
# -------------------------
print("\n📖 완료된 샘플에 대응하는 GT만 계산...")
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    lines = f.readlines()

pairs = []
for line in tqdm(lines, desc="GT 계산"):
    ex = json.loads(line)
    msgs = ex["messages"]

    user_msg = next(m for m in msgs if m["role"] == "user")
    asst_msg = next((m for m in msgs if m["role"] == "assistant"), None)
    if asst_msg is None:
        continue

    user_text = parse_user_text(user_msg["content"])
    m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
    m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
    if not m_ch or not m_vn:
        continue

    key = (m_ch.group(1).strip(), m_vn.group(1).strip())
    if key not in done_map:
        continue

    asst_text = asst_msg["content"] if isinstance(asst_msg["content"], str) \
                else parse_user_text(asst_msg["content"])

    try:
        gt_instances = json.loads(asst_text).get("instances", [])
    except:
        gt_instances = []

    pairs.append({
        "channel":   key[0],
        "video":     key[1],
        "gt_tok":    count_tokens(asst_text),
        "pred_tok":  done_map[key]["pred_tok"],
        "gt_inst":   len(gt_instances),
        "pred_inst": done_map[key]["pred_inst"],
    })

print(f"✅ 매칭된 페어: {len(pairs)}")

# -------------------------
# 4~7) 기존 통계 (토큰 기준)
# -------------------------
gt_lens   = np.array([p["gt_tok"]   for p in pairs])
pred_lens = np.array([p["pred_tok"] for p in pairs])

def print_stats(arr, name):
    print(f"\n📊 {name}")
    print(f"  개수:    {len(arr)}")
    print(f"  mean:    {arr.mean():.1f}")
    print(f"  median:  {np.median(arr):.1f}")
    print(f"  std:     {arr.std():.1f}")
    print(f"  min:     {arr.min()}")
    print(f"  max:     {arr.max()}")
    print(f"  p25:     {np.percentile(arr, 25):.1f}")
    print(f"  p75:     {np.percentile(arr, 75):.1f}")
    print(f"  p95:     {np.percentile(arr, 95):.1f}")
    print(f"  p99:     {np.percentile(arr, 99):.1f}")

print_stats(gt_lens,   "GT 토큰수 (완료된 것만)")
print_stats(pred_lens, "Pred 토큰수")

print("\n📋 GT vs Pred 토큰수 비교 (같은 샘플끼리)")
print(f"  {'metric':<10} {'GT':>12} {'Pred':>12} {'diff':>12}")
print(f"  {'-'*10} {'-'*12} {'-'*12} {'-'*12}")
for label, fn in [
    ("mean",   np.mean),
    ("median", np.median),
    ("std",    np.std),
    ("min",    np.min),
    ("max",    np.max),
    ("p25",    lambda a: np.percentile(a, 25)),
    ("p75",    lambda a: np.percentile(a, 75)),
    ("p95",    lambda a: np.percentile(a, 95)),
    ("p99",    lambda a: np.percentile(a, 99)),
]:
    g, p = fn(gt_lens), fn(pred_lens)
    print(f"  {label:<10} {g:>12.1f} {p:>12.1f} {p-g:>+12.1f}")

diff = pred_lens - gt_lens
print("\n📐 샘플별 토큰 차이 (Pred - GT)")
print(f"  mean:   {diff.mean():+.1f}")
print(f"  median: {np.median(diff):+.1f}")
print(f"  std:    {diff.std():.1f}")
print(f"  min:    {diff.min():+.0f}")
print(f"  max:    {diff.max():+.0f}")
print(f"  Pred가 더 긴 샘플: {(diff > 0).sum()} ({100*(diff > 0).mean():.1f}%)")
print(f"  Pred가 더 짧은 샘플: {(diff < 0).sum()} ({100*(diff < 0).mean():.1f}%)")
print(f"  완전 동일: {(diff == 0).sum()}")

ratio = pred_lens / np.maximum(gt_lens, 1)
print(f"\n📏 토큰 비율 (Pred / GT)")
print(f"  median: {np.median(ratio):.3f}")
print(f"  p25:    {np.percentile(ratio, 25):.3f}")
print(f"  p75:    {np.percentile(ratio, 75):.3f}")
print(f"  < 0.5:  {(ratio < 0.5).sum():>6} ({100*(ratio < 0.5).mean():.1f}%)  ← Pred이 너무 짧음")
print(f"  < 0.8:  {(ratio < 0.8).sum():>6} ({100*(ratio < 0.8).mean():.1f}%)")
print(f"  0.8~1.2:{((ratio >= 0.8) & (ratio <= 1.2)).sum():>6} ({100*((ratio >= 0.8) & (ratio <= 1.2)).mean():.1f}%)  ← 유사한 토큰수")
print(f"  > 1.2:  {(ratio > 1.2).sum():>6} ({100*(ratio > 1.2).mean():.1f}%)")
print(f"  > 2.0:  {(ratio > 2.0).sum():>6} ({100*(ratio > 2.0).mean():.1f}%)  ← Pred이 너무 김")

# -------------------------
# 8) GT 인스턴스 수 구간별 비교
# -------------------------
gt_insts  = np.array([p["gt_inst"]   for p in pairs])
pred_insts = np.array([p["pred_inst"] for p in pairs])

print("\n📊 인스턴스 수 분포 비교")
print(f"  {'구간':<20} {'GT 샘플수':>10} {'Pred 샘플수':>12} {'GT %':>8} {'Pred %':>8}")
print(f"  {'-'*20} {'-'*10} {'-'*12} {'-'*8} {'-'*8}")

inst_bins = [0, 1, 5, 10, 20, 50, 100, 200, 500, 10000]
for i in range(len(inst_bins) - 1):
    lo, hi = inst_bins[i], inst_bins[i+1]
    g_cnt = int(((gt_insts >= lo) & (gt_insts < hi)).sum())
    p_cnt = int(((pred_insts >= lo) & (pred_insts < hi)).sum())
    g_pct = g_cnt / len(gt_insts) * 100
    p_pct = p_cnt / len(pred_insts) * 100
    print(f"  {lo:>6} ~ {hi:<10} {g_cnt:>10,} {p_cnt:>12,} {g_pct:>7.1f}% {p_pct:>7.1f}%")

# -------------------------
# 9) GT 토큰 구간별 인스턴스 수 비교
# -------------------------
print("\n📊 토큰 수 분포 비교")
print(f"  {'구간':<20} {'GT 샘플수':>10} {'Pred 샘플수':>12} {'GT %':>8} {'Pred %':>8}")
print(f"  {'-'*20} {'-'*10} {'-'*12} {'-'*8} {'-'*8}")

tok_bins = [0, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 65536]
for i in range(len(tok_bins) - 1):
    lo, hi = tok_bins[i], tok_bins[i+1]
    g_cnt = int(((gt_lens >= lo) & (gt_lens < hi)).sum())
    p_cnt = int(((pred_lens >= lo) & (pred_lens < hi)).sum())
    g_pct = g_cnt / len(gt_lens) * 100
    p_pct = p_cnt / len(pred_lens) * 100
    print(f"  {lo:>6} ~ {hi:<10} {g_cnt:>10,} {p_cnt:>12,} {g_pct:>7.1f}% {p_pct:>7.1f}%")

🔧 tokenizer 로딩...


The tokenizer you are loading from '/home/taeyoung/nfs-mount/chi2027/model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



📦 완료된 생성 JSON 수집...


생성 JSON 파싱: 100%|██████████| 6606/6606 [00:01<00:00, 5647.99it/s]


✅ 완료된 생성물: 3299

📖 완료된 샘플에 대응하는 GT만 계산...


GT 계산: 100%|██████████| 3320/3320 [00:05<00:00, 629.45it/s]

✅ 매칭된 페어: 3304

📊 GT 토큰수 (완료된 것만)
  개수:    3304
  mean:    1325.5
  median:  754.0
  std:     2155.4
  min:     4
  max:     47561
  p25:     113.5
  p75:     1691.0
  p95:     4386.1
  p99:     9018.3

📊 Pred 토큰수
  개수:    3304
  mean:    1311.8
  median:  618.0
  std:     2105.0
  min:     5
  max:     19898
  p25:     65.0
  p75:     1342.5
  p95:     5143.2
  p99:     10826.1

📋 GT vs Pred 토큰수 비교 (같은 샘플끼리)
  metric               GT         Pred         diff
  ---------- ------------ ------------ ------------
  mean             1325.5       1311.8        -13.6
  median            754.0        618.0       -136.0
  std              2155.4       2105.0        -50.4
  min                 4.0          5.0         +1.0
  max             47561.0      19898.0     -27663.0
  p25               113.5         65.0        -48.5
  p75              1691.0       1342.5       -348.5
  p95              4386.1       5143.2       +757.2
  p99              9018.3      10826.1      +1807.8

📐 샘플별 토큰 차이 (P

In [11]:
# %% 미완료 채널별 현황
import os, re, json
from collections import defaultdict
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test"

def parse_user_text(msg_content):
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(part.get("text", "") for part in msg_content if isinstance(part, dict))
    return ""

# eval 전체 파싱
with open(EVAL_JSONL, "r") as f:
    eval_lines = f.readlines()

# 채널별 전체/완료/미완료
channel_total = defaultdict(int)
channel_done = defaultdict(int)
channel_missing = defaultdict(list)

for line in eval_lines:
    ex = json.loads(line)
    msgs = ex["messages"]
    user_msg = next(m for m in msgs if m["role"] == "user")
    user_text = parse_user_text(user_msg["content"])
    m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
    m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
    if not m_ch or not m_vn:
        continue
    ch = m_ch.group(1).strip()
    vn = m_vn.group(1).strip()

    # 본채널 + swap 두 건
    for cond_ch_path in [os.path.join(OUTPUT_DIR, ch, vn, f"{ch}.json")]:
        channel_total[ch] += 1
        if os.path.exists(cond_ch_path):
            channel_done[ch] += 1
        else:
            channel_missing[ch].append(vn)

    # swap도 확인
    swap_dir = os.path.join(OUTPUT_DIR, ch, vn)
    if os.path.isdir(swap_dir):
        swap_files = [f for f in os.listdir(swap_dir) if f.endswith(".json") and f != f"{ch}.json"]
        if swap_files:
            channel_done[ch] += 1
            channel_total[ch] += 1
        else:
            channel_total[ch] += 1
    else:
        channel_total[ch] += 1

# 미완료만 출력
incomplete = {ch: len(vids) for ch, vids in channel_missing.items() if len(vids) > 0}
incomplete_sorted = sorted(incomplete.items(), key=lambda x: x[1], reverse=True)

total_missing = sum(v for v in incomplete.values())
total_all = sum(channel_total.values())
total_done_all = sum(channel_done.values())

print(f"전체: {total_all:,}건, 완료: {total_done_all:,}건, 미완료: {total_missing:,}건")
print(f"미완료 채널: {len(incomplete)}개\n")

print(f"{'채널':<40} {'미완료':>6} {'전체':>6}")
print(f"{'─'*40} {'─'*6} {'─'*6}")
for ch, cnt in incomplete_sorted:
    print(f"  {ch:<40} {cnt:>5} {channel_total[ch]:>5}")

전체: 6,640건, 완료: 6,614건, 미완료: 17건
미완료 채널: 11개

채널                                          미완료     전체
──────────────────────────────────────── ────── ──────
  KBS 슈퍼맨이 돌아왔다                                4    10
  요올리Yoli                                      4    10
  CJ ENM                                       1    10
  LCK                                          1    10
  LeoJ Makeup                                  1    10
  Marine Ch. 宝鐘マリン                             1    10
  T1                                           1    10
  대도서관TV (buzzbean11)                          1    10
  보겸TV                                         1    10
  보물섬                                          1    10
  해피팸(Happy family)                            1    10


In [13]:
# %% 미완료 채널별 현황
import os, re, json
from collections import defaultdict
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test"

def parse_user_text(msg_content):
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(part.get("text", "") for part in msg_content if isinstance(part, dict))
    return ""

# eval 전체 파싱
with open(EVAL_JSONL, "r") as f:
    eval_lines = f.readlines()

# 채널별 전체/완료/미완료
channel_total = defaultdict(int)
channel_done = defaultdict(int)
channel_missing = defaultdict(list)

for line in eval_lines:
    ex = json.loads(line)
    msgs = ex["messages"]
    user_msg = next(m for m in msgs if m["role"] == "user")
    user_text = parse_user_text(user_msg["content"])
    m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
    m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
    if not m_ch or not m_vn:
        continue
    ch = m_ch.group(1).strip()
    vn = m_vn.group(1).strip()

    # 본채널 + swap 두 건
    for cond_ch_path in [os.path.join(OUTPUT_DIR, ch, vn, f"{ch}.json")]:
        channel_total[ch] += 1
        if os.path.exists(cond_ch_path):
            channel_done[ch] += 1
        else:
            channel_missing[ch].append(vn)

    # swap도 확인
    swap_dir = os.path.join(OUTPUT_DIR, ch, vn)
    if os.path.isdir(swap_dir):
        swap_files = [f for f in os.listdir(swap_dir) if f.endswith(".json") and f != f"{ch}.json"]
        if swap_files:
            channel_done[ch] += 1
            channel_total[ch] += 1
        else:
            channel_total[ch] += 1
    else:
        channel_total[ch] += 1

# 미완료만 출력
incomplete = {ch: len(vids) for ch, vids in channel_missing.items() if len(vids) > 0}
incomplete_sorted = sorted(incomplete.items(), key=lambda x: x[1], reverse=True)

total_missing = sum(v for v in incomplete.values())
total_all = sum(channel_total.values())
total_done_all = sum(channel_done.values())

print(f"전체: {total_all:,}건, 완료: {total_done_all:,}건, 미완료: {total_missing:,}건")
print(f"미완료 채널: {len(incomplete)}개\n")

print(f"{'채널':<40} {'미완료':>6} {'전체':>6}")
print(f"{'─'*40} {'─'*6} {'─'*6}")
for ch, cnt in incomplete_sorted:
    print(f"  {ch:<40} {cnt:>5} {channel_total[ch]:>5}")

전체: 6,640건, 완료: 6,616건, 미완료: 16건
미완료 채널: 10개

채널                                          미완료     전체
──────────────────────────────────────── ────── ──────
  KBS 슈퍼맨이 돌아왔다                                4    10
  요올리Yoli                                      4    10
  CJ ENM                                       1    10
  LCK                                          1    10
  LeoJ Makeup                                  1    10
  Marine Ch. 宝鐘マリン                             1    10
  대도서관TV (buzzbean11)                          1    10
  보겸TV                                         1    10
  보물섬                                          1    10
  해피팸(Happy family)                            1    10


In [2]:
# %% 최근 완료 파일 26개
import os, glob, json
from datetime import datetime

OUTPUT_DIR = "./data/7_qwen_test"

files = glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True)

# 수정시간 기준 정렬
files_with_mtime = [(f, os.path.getmtime(f)) for f in files]
files_with_mtime.sort(key=lambda x: x[1], reverse=True)

print(f"전체 파일: {len(files_with_mtime):,}개\n")
print(f"{'#':>3} {'시간':>20} {'채널/영상/조건':50} {'토큰':>8}")
print(f"{'─'*3} {'─'*20} {'─'*50} {'─'*8}")

for i, (path, mtime) in enumerate(files_with_mtime[:24]):
    ts = datetime.fromtimestamp(mtime).strftime("%m-%d %H:%M:%S")
    try:
        with open(path, "r") as f:
            rec = json.load(f)
        tok = rec.get("completion_tokens", "?")
    except:
        tok = "?"
    
    rel = os.path.relpath(path, OUTPUT_DIR)
    print(f"  {i+1:>2} {ts:>20} {rel[:100]:<50} {tok:>8}")

전체 파일: 6,630개

  #                   시간 채널/영상/조건                                                 토큰
─── ──────────────────── ────────────────────────────────────────────────── ────────
   1       04-22 12:24:43 Marine Ch. 宝鐘マリン/実録！女子校あるある8選wwwwww【ホロライブ⧸宝鐘マリン】#shorts/Marine Ch. 宝鐘マリン.json     1845
   2       04-22 12:24:39 요올리Yoli/미국식 핫도그🌭 말랑이 만들기 - DIY Hot dog Squishy with nano tape #shorts/요올리Yoli.json     1511
   3       04-22 12:21:55 요올리Yoli/포도🍇 테이프공 만들기 -DIY Grape Nano Tape Bubbles!/요올리Yoli.json     1564
   4       04-22 12:21:45 요올리Yoli/도어즈🚪스크리치 테이프공 만들기 - DIY DOORS Screech Nano Tape Bubbles!/요올리Yoli.json      913
   5       04-22 12:21:44 KBS 슈퍼맨이 돌아왔다/절대 지용 오빠를 귀찮게 해서는 안 돼💦 #추사랑 #GD #GDRAGON #지드래곤 #지디 #권지용 #사랑이 #슈돌 #슈퍼맨이 돌아왔다/KBS 슈퍼맨이 돌      820
   6       04-22 12:21:41 LCK/＂쓰레기 같은 놈＂/LCK.json                                 603
   7       04-22 12:09:32 레시피 읽어주는 여자/용암만들기 튜토리얼/Victor Entertainment.json       1354
   8       04-22 12:09:31 딩고 스튜디오 _ dingo studio/개웃긴 홍석천 댓글읽기&대댓

In [1]:
# %% 최근 파일 5개 삭제
import os, glob
from datetime import datetime

OUTPUT_DIR = "./data/7_qwen_test"

files = glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True)
files_with_mtime = [(f, os.path.getmtime(f)) for f in files]
files_with_mtime.sort(key=lambda x: x[1], reverse=True)

for f, mtime in files_with_mtime[:7]:
    ts = datetime.fromtimestamp(mtime).strftime("%m-%d %H:%M:%S")
    rel = os.path.relpath(f, OUTPUT_DIR)
    os.remove(f)
    print(f"  삭제: {ts} {rel}")

print(f"\n✅ {len(files_with_mtime)}개 삭제 완료")

  삭제: 04-22 12:07:05 SBS Radio 에라오/[선공개] 라이브도 잘하는 갓기들의 'Hype Boy' 무반주 한소절🎤 ｜ 뉴진스(NewJeans) ｜ 두시탈출 컬투쇼/Chogakusei Official.json
  삭제: 04-22 12:07:02 요올리Yoli/포도🍇 테이프공 만들기 -DIY Grape Nano Tape Bubbles!/요올리Yoli.json
  삭제: 04-22 12:06:59 요올리Yoli/미국식 핫도그🌭 말랑이 만들기 - DIY Hot dog Squishy with nano tape #shorts/요올리Yoli.json
  삭제: 04-22 12:06:58 LCK/＂쓰레기 같은 놈＂/LCK.json
  삭제: 04-22 12:06:55 LeoJ Makeup/사고 났을 때 흔한 T반응/대생이.json
  삭제: 04-22 12:06:55 Marine Ch. 宝鐘マリン/実録！女子校あるある8選wwwwww【ホロライブ⧸宝鐘マリン】#shorts/Marine Ch. 宝鐘マリン.json
  삭제: 04-22 12:06:54 LeoJ Makeup/사고 났을 때 흔한 T반응/LeoJ Makeup.json

✅ 6617개 삭제 완료
